# 任务二：人脸表情识别 · Kaggle 历史最高分四模型融合

本 notebook 默认使用 **ConvNeXt-Tiny + QCS + EmoNeXt-Tiny + EmotiEff-B2**，四模型各占 0.25。
完整保留模型、数据、训练、损失、评估、推理和提交代码。

| 版本 | 固定开发验证 Accuracy | 固定开发验证 Macro-F1 | 已提交 Kaggle Public Score |
|---|---:|---:|---:|
| **默认：72.38% 验证版** | **72.3794%** | **70.7087%** | **0.742** |
| 72.53% 验证版 | 72.5273% | 70.8470% | 0.740 |

历史比赛分数来自账号已完成的两次提交；新运行的成绩仍以本次 Kaggle 评测为准。
默认按已知比赛分数选择 0.742 版。验证集用于调参、早停和方案选择，验证成绩属于开发验证结果。

**执行流程**：核验私有输入 → 展开 39 个源码文件 → 核验检查点 → FP32 四模型推理 →
复核固定验证集证据 → 按比赛类别映射与原始 ID 顺序生成 `submission.csv`。
模型依次加载，使用 FP32，关闭 TF32 和水平翻转 TTA。默认不重复本地训练。


## 1. 环境与私有输入

输入资源包含四个完整最佳检查点、固定数据缓存、逐模型验证概率与 SHA-256 清单。
所有图像缓存由本比赛已提供的数据生成。测试预测仅由模型推理得到。
默认在有 GPU 时重新推理；CPU 会明确使用已保存的最佳概率，生成同一份已核验提交。

默认方案使用基础资源加 public-best 补充资源；补充资源只增加两个不同的检查点和其预测记录。


In [ ]:
from pathlib import Path
import hashlib, json, sys, subprocess, gc, time, os, shutil
import numpy as np
import pandas as pd
import torch

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'kaggle_local_check'
WORK.mkdir(parents=True, exist_ok=True)
input_root = Path(os.environ.get('FER_BUNDLE_ROOT', '/kaggle/input'))
manifests = sorted(input_root.rglob('fer_bundle_manifest.json'))
if not manifests:
    # Supports Kaggle inputs that retain the original uploaded archive.
    import zipfile
    archives = sorted(input_root.rglob('fer_best_7253_assets.zip'))
    assert len(archives) == 1, f'Expected a FER bundle manifest or one archive; found: {archives}'
    destination = (WORK / 'fer_bundle_input').resolve()
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            resolved = (destination / member.filename).resolve()
            assert resolved.is_relative_to(destination), f'Invalid archive path: {member.filename}'
        archive.extractall(destination)
    manifests = sorted(destination.rglob('fer_bundle_manifest.json'))
assert len(manifests) == 1, f'Expected exactly one FER bundle, found: {manifests}'
BUNDLE = manifests[0].parent
BUNDLE_MANIFEST = json.loads(manifests[0].read_text(encoding='utf-8'))

ACTIVE_VERSION = '7238'  # '7253' selects the alternative with higher development validation accuracy.
FILE_SPECS = dict(BUNDLE_MANIFEST['files'])
FILE_ROOTS = {relative: BUNDLE for relative in FILE_SPECS}
if ACTIVE_VERSION == '7238':
    supplement_root = Path(os.environ.get('FER_SUPPLEMENT_ROOT', str(input_root)))
    supplements = sorted(supplement_root.rglob('fer_public_best_manifest.json'))
    assert len(supplements) == 1, f'Expected the public-best supplement; found: {supplements}'
    SUPPLEMENT = supplements[0].parent
    supplement_manifest = json.loads(supplements[0].read_text(encoding='utf-8'))
    assert supplement_manifest['manifest_sha256'] == BUNDLE_MANIFEST['manifest_sha256']
    FILE_SPECS.update(supplement_manifest['files'])
    FILE_ROOTS.update({relative: SUPPLEMENT for relative in supplement_manifest['files']})
    BUNDLE_MANIFEST = supplement_manifest
PROJECT = WORK / 'fer_project'
PROJECT.mkdir(exist_ok=True)
RECOMPUTE_TEST = torch.cuda.is_available()
RECOMPUTE_VALIDATION = False  # True adds four-model inference on the fixed validation split.
BATCH_SIZE = 32
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('Bundle:', BUNDLE)
print('PyTorch:', torch.__version__, '| device:', DEVICE)
print('Test probability source:', 'recomputed checkpoints' if RECOMPUTE_TEST else 'verified saved model probabilities')

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def checked_file(relative):
    path = FILE_ROOTS[relative] / relative
    spec = FILE_SPECS[relative]
    assert path.stat().st_size == spec['bytes'], f'File size mismatch: {relative}'
    assert file_sha256(path) == spec['sha256'], f'SHA-256 mismatch: {relative}'
    return path


In [ ]:
# Use the reference timm release without changing Kaggle's torch installation.
import importlib.metadata
try:
    installed_timm = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError:
    installed_timm = None
if installed_timm != '1.0.29':
    wheels = list((BUNDLE / 'wheels').glob('timm-1.0.29-*.whl'))
    assert len(wheels) == 1, 'The pinned offline timm wheel is missing.'
    checked_file('wheels/' + wheels[0].name)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheels[0])])
import timm
assert timm.__version__ == '1.0.29'
print('timm:', timm.__version__)

def write_sources(files):
    for relative, text in files.items():
        path = PROJECT / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding='utf-8')
    print('Project modules:', ', '.join(files))

sys.path.insert(0, str(PROJECT))


## 2. 模型实现与来源

下列代码单元保存完整项目模块，以保留原有相对导入和训练接口。
`write_sources` 中的每个键就是模块路径，值是完整 Python 源码。
QCS 只增加“直接加载完整最佳检查点时跳过 IR50 初始化文件”的适配；结构和推理不变。
每次加载仍使用 `strict=True` 校验全部模型张量。

来源及固定提交记录在 `third_party/SOURCES.json`，QCS 保留原 MIT LICENSE。
原始实现：
[EmoNeXt](https://github.com/yelboudouri/EmoNeXt/tree/ce77dc13e13d3b09742f0f4efb9dd83fcf3b6c95)、
[QCS](https://github.com/birdwcp/QCS/tree/293f28f948dccdca385917fbc0cfeafd6220d2e2)、
[EmotiEffLib](https://github.com/sb-ai-lab/EmotiEffLib/tree/520a051c64cd191521e5934655314e769a319684)。


In [ ]:
write_sources({
'third_party/__init__.py': """\
# Local FER experiment package.
""",

'third_party/emonext/__init__.py': """\
# Local FER experiment package.
""",

'third_party/emonext/models.py': """\
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.hub import load_state_dict_from_url
from torchvision.models import vgg16_bn, VGG16_BN_Weights
from torchvision.ops import StochasticDepth

model_urls = {
    "convnext_tiny_1k": "https://dl.fbaipublicfiles.com/convnext/convnext_tiny_1k_224_ema.pth",
    "convnext_small_1k": "https://dl.fbaipublicfiles.com/convnext/convnext_small_1k_224_ema.pth",
    "convnext_base_1k": "https://dl.fbaipublicfiles.com/convnext/convnext_base_1k_224_ema.pth",
    "convnext_large_1k": "https://dl.fbaipublicfiles.com/convnext/convnext_large_1k_224_ema.pth",
    "convnext_tiny_22k": "https://dl.fbaipublicfiles.com/convnext/convnext_tiny_22k_224.pth",
    "convnext_small_22k": "https://dl.fbaipublicfiles.com/convnext/convnext_small_22k_224.pth",
    "convnext_base_22k": "https://dl.fbaipublicfiles.com/convnext/convnext_base_22k_224.pth",
    "convnext_large_22k": "https://dl.fbaipublicfiles.com/convnext/convnext_large_22k_224.pth",
    "convnext_xlarge_22k": "https://dl.fbaipublicfiles.com/convnext/convnext_xlarge_22k_224.pth",
}


class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class DotProductSelfAttention(nn.Module):
    def __init__(self, input_dim):
        super(DotProductSelfAttention, self).__init__()
        self.input_dim = input_dim
        self.norm = nn.LayerNorm(input_dim)
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)

    def forward(self, x):
        x = self.norm(x)
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)

        scale = 1 / math.sqrt(math.sqrt(self.input_dim))
        scores = torch.matmul(query, key.transpose(-2, -1)) * scale
        attention_weights = torch.softmax(scores, dim=-1)

        attended_values = torch.matmul(attention_weights, value)
        output = attended_values + x

        return output, attention_weights


class LayerNorm(nn.Module):
    r\"\"\"LayerNorm that supports two data formats: channels_last (default) or channels_first.
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs
    with shape (batch_size, channels, height, width).
    \"\"\"

    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(
                x, self.normalized_shape, self.weight, self.bias, self.eps
            )
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x


class Block(nn.Module):
    r\"\"\"ConvNeXt Block. There are two equivalent implementations:
    (1) DwConv -> LayerNorm (channels_first) -> 1x1 Conv -> GELU -> 1x1 Conv; all in (N, C, H, W)
    (2) DwConv -> Permute to (N, H, W, C); LayerNorm (channels_last) -> Linear -> GELU -> Linear; Permute back
    We use (2) as we find it slightly faster in PyTorch

    Args:
        dim (int): Number of input channels.
        drop_path (float): Stochastic depth rate. Default: 0.0
        layer_scale_init_value (float): Init value for Layer Scale. Default: 1e-6.
    \"\"\"

    def __init__(self, dim, drop_path=0.0, layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(
            dim, dim, kernel_size=7, padding=3, groups=dim
        )  # depthwise conv
        self.norm = LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(
            dim, 4 * dim
        )  # pointwise/1x1 convs, implemented with linear layers
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = (
            nn.Parameter(layer_scale_init_value * torch.ones((dim)), requires_grad=True)
            if layer_scale_init_value > 0
            else None
        )
        self.stochastic_depth = StochasticDepth(drop_path, "row")

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)  # (N, C, H, W) -> (N, H, W, C)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)

        x = input + self.stochastic_depth(x)
        return x


class EmoNeXt(nn.Module):
    def __init__(
        self,
        in_chans=3,
        num_classes=1000,
        depths=None,
        dims=None,
        drop_path_rate=0.0,
        layer_scale_init_value=1e-6,
    ):
        super().__init__()

        if dims is None:
            dims = [96, 192, 384, 768]
        if depths is None:
            depths = [3, 3, 9, 3]

        # Spatial transformer localization-network
        self.localization = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=7),
            nn.BatchNorm2d(8),
            nn.MaxPool2d(2, stride=2),
            nn.ReLU(True),
            nn.Conv2d(8, 10, kernel_size=5),
            nn.BatchNorm2d(10),
            nn.MaxPool2d(2, stride=2),
            nn.ReLU(True),
        )

        # Regressor for the 3 * 2 affine matrix
        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 52 * 52, 32), nn.ReLU(True), nn.Linear(32, 3 * 2)
        )

        self.downsample_layers = (
            nn.ModuleList()
        )  # stem and 3 intermediate downsampling conv layers
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            LayerNorm(dims[0], eps=1e-6, data_format="channels_first"),
        )
        self.downsample_layers.append(stem)
        for i in range(3):
            downsample_layer = nn.Sequential(
                LayerNorm(dims[i], eps=1e-6, data_format="channels_first"),
                nn.Conv2d(dims[i], dims[i + 1], kernel_size=2, stride=2),
                SELayer(dims[i + 1]),
            )
            self.downsample_layers.append(downsample_layer)

        self.stages = (
            nn.ModuleList()
        )  # 4 feature resolution stages, each consisting of multiple residual blocks
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i in range(4):
            stage = nn.Sequential(
                *[
                    Block(
                        dim=dims[i],
                        drop_path=dp_rates[cur + j],
                        layer_scale_init_value=layer_scale_init_value,
                    )
                    for j in range(depths[i])
                ]
            )
            self.stages.append(stage)
            cur += depths[i]

        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)  # final norm layer
        self.attention = DotProductSelfAttention(dims[-1])
        self.head = nn.Linear(dims[-1], num_classes)

        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(
            torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float)
        )

    def stn(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 52 * 52)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)

        grid = F.affine_grid(theta, x.size(), align_corners=True)
        x = F.grid_sample(x, grid, align_corners=True)

        return x

    def forward_features(self, x):
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        return self.norm(
            x.mean([-2, -1])
        )  # global average pooling, (N, C, H, W) -> (N, C)

    def forward(self, x, labels=None):
        x = self.stn(x)
        x = self.forward_features(x)
        _, weights = self.attention(x)
        logits = self.head(x)

        if labels is not None:
            mean_attention_weight = torch.mean(weights)
            attention_loss = torch.mean((weights - mean_attention_weight) ** 2)

            loss = F.cross_entropy(logits, labels, label_smoothing=0.2) + attention_loss
            return torch.argmax(logits, dim=1), logits, loss

        return torch.argmax(logits, dim=1), logits


def get_model(num_classes, model_size="tiny", in_22k=False):
    if model_size == "tiny":
        depths = [3, 3, 9, 3]
        dims = [96, 192, 384, 768]
        url = (
            model_urls["convnext_tiny_22k"]
            if in_22k
            else model_urls["convnext_tiny_1k"]
        )
    elif model_size == "small":
        depths = [3, 3, 27, 3]
        dims = [96, 192, 384, 768]
        url = (
            model_urls["convnext_small_22k"]
            if in_22k
            else model_urls["convnext_small_1k"]
        )
    elif model_size == "base":
        depths = [3, 3, 27, 3]
        dims = [128, 256, 512, 1024]
        url = (
            model_urls["convnext_base_22k"]
            if in_22k
            else model_urls["convnext_base_1k"]
        )
    elif model_size == "large":
        depths = [3, 3, 27, 3]
        dims = [192, 384, 768, 1536]
        url = (
            model_urls["convnext_large_22k"]
            if in_22k
            else model_urls["convnext_large_1k"]
        )
    else:
        depths = [3, 3, 27, 3]
        dims = [256, 512, 1024, 2048]
        url = model_urls["convnext_xlarge_22k"]

    default_num_classes = 1000
    if in_22k:
        default_num_classes = 21841

    net = EmoNeXt(
        depths=depths, dims=dims, num_classes=default_num_classes, drop_path_rate=0.1
    )

    state_dict = load_state_dict_from_url(url=url)
    net.load_state_dict(state_dict["model"], strict=False)
    net.head = nn.Linear(dims[-1], num_classes)

    return net
""",

'third_party/nla/__init__.py': """\
# Local FER experiment package.
""",

'third_party/nla/loss.py': """\
import torch.nn.functional as F
import torch.nn as nn
import torch
import math
import numpy as np

# Noise-aware-Adaptive Weight 
class NAW(nn.Module):
    def __init__(self, args, mu_x_t, mu_y_t=0.50, mu_x_f=0.30, mu_y_f=0.15, f_minor=0.80, f_major=0.85, t_minor=-0.50, t_major=0.80, t_lambda=1.0):
        super(NAW, self).__init__()
        t_minor = args.eps * t_minor
        self.t_lambda = t_lambda
        self.t_mu = torch.tensor([mu_x_t, mu_y_t],dtype=torch.float).unsqueeze(1).to('cuda')
        self.f_mu = torch.tensor([mu_x_f, mu_y_f],dtype=torch.float).unsqueeze(1).to('cuda')
        self.t_cov = torch.tensor([[t_major, t_minor], [t_minor, t_major]],dtype=torch.float).to('cuda')
        self.f_cov = torch.tensor([[f_major, f_minor], [f_minor, f_major]],dtype=torch.float).to('cuda')



    def forward(self, inputs, targets):
        probs = torch.softmax(inputs, dim=1)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        P_GT = probs.gather(1, targets.unsqueeze(1)).squeeze()

        top2_probs = probs.topk(2, dim=1).values
        max_probs = top2_probs[:, 0]
        second_max_probs = top2_probs[:, 1]

        P_MAX_is_GT = max_probs == P_GT

        P_NN = torch.where(P_MAX_is_GT, second_max_probs, max_probs)
        SV_flat = torch.stack([P_GT, P_NN], axis=0)

        t_value = self.t_lambda*self.co_gau(SV_flat, self.t_cov, self.t_mu) 
        f_value = self.co_gau(SV_flat, self.f_cov, self.f_mu)

        value = torch.where(P_MAX_is_GT, t_value, f_value)
        Adaptive_loss = ce_loss * value

        return Adaptive_loss


    def co_gau(self, SV_flat, cov, mu):
        diff = SV_flat - mu  # [2, N]
        inv_cov = torch.linalg.inv(cov)  # [2, 2]
        det_cov = torch.det(cov)  # scalar

        exponent = -0.5 * torch.sum(diff * (inv_cov @ diff), dim=0)

        k = SV_flat.shape[0]  
        norm_factor = 1.0 / torch.sqrt((2 * torch.tensor(math.pi)) ** k * det_cov)

        value = norm_factor * torch.exp(exponent)  
        return value





# JSD regularizer
def jensen_shannon_divergence(p, q):
    m = 0.5 * (p + q)
    kl_div_p = F.kl_div(F.log_softmax(p, dim=1), F.softmax(m, dim=1), reduction='sum')
    kl_div_q = F.kl_div(F.log_softmax(q, dim=1), F.softmax(m, dim=1), reduction='sum')
    js_divergence = 0.5 * (kl_div_p + kl_div_q)
    return js_divergence


# sigma scheduler
def exponential_scheduler(epoch, epochs, slope=-15, sch_bool=True):
    if sch_bool:
        return (1 - np.exp(slope * epoch / epochs))
    else:
        return 1
""",

'third_party/nla/main_rafdb.py': """\
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "3" 
import numpy as np
import argparse
import torch
import torch.nn as nn
from torchvision import transforms
import wandb
from src.dataset import NLA_Rafdb
from src.model import NLA_r18
from src.loss import * 
from src.utils import *
from src.resnet import *
from src.test import *
from src.train import *
torch.autograd.set_detect_anomaly(True)

parser = argparse.ArgumentParser()
parser.add_argument('--dataset', default="rafdb", type=str, choices=['rafdb', 'Ferplus', 'AffecNet'], help="experiment dataset")
parser.add_argument('--exp_name', default="NAW_NLA", type=str, choices=['NAW_NLA', 'NAW_CAM', 'NAW_L1', 'L1', 'CAM', 'JSD'], help="training strategy")
parser.add_argument('--exp_ver', default="temp", type=str, help="experiment version and date")
parser.add_argument('--dataset_path', type=str, default='/workspace/NLA/imgs/RAF-DB', help='raf_dataset_path')
parser.add_argument('--label_path', type=str, default='/workspace/NLA/imgs/list_patition_label.txt', help='label_path')
parser.add_argument('--num_classes', type=int, default=7)
parser.add_argument('--feature_embedding', type=int, default=512)
parser.add_argument('--output', default="/workspace/NLA/AAAI", type=str, help="output dir")
parser.add_argument('--batch_size', type=int, default=32, help='batch_size')
parser.add_argument('--lr', type=float, default=0.0001, help='lr')
parser.add_argument('--workers', type=int, default=4, help='number of workers')
parser.add_argument('--gpu', type=int, default=0, help='the number of the device')
parser.add_argument('--epochs', type=int, default=60, help='number of epochs')
parser.add_argument('--save_freq', type=int, default=5, help='save frequency')  
# other setting
parser.add_argument('--noise', type=bool, default=False, help='learning from noise label')  
parser.add_argument('--noise_name', type=str, default="10_1", help='noise percentage_random seed')  
parser.add_argument('--imbalanced', type=bool, default=False, help='learning from imbalanced label')
parser.add_argument('--imbalanced_name', type=str, default="20_256", help='imbalance factor_random seed')  
parser.add_argument('--seed', type=int, default=0) # 11111135
parser.add_argument('--lam_a', type=float, default=0.5)
parser.add_argument('--lam_b', type=float, default=0.5)
parser.add_argument('--lam_c', type=float, default=0.5)
parser.add_argument('--slope', type=float, default=-15)
parser.add_argument('--t_lambda', type=float, default=1)
parser.add_argument('--sch_bool', type=bool, default=True)
parser.add_argument('--mu_x_t', type=float, default=0.5) 
parser.add_argument('--mu_y_t', type=float, default=0.5) 
parser.add_argument('--mu_x_f', type=float, default=0.30) 
parser.add_argument('--mu_y_f', type=float, default=0.15) 
parser.add_argument('--t_std_major', type=float, default=0.8) 
parser.add_argument('--t_std_ratio', type=float, default=2) 
parser.add_argument('--f_std_major', type=float, default=0.91) 
parser.add_argument('--f_std_ratio', type=float, default=6) 
args = parser.parse_args()


def main():    
    setup_seed(args.seed)
    args.con = 0
    args.exp_name = args.exp_name
    args.output = os.path.join(args.output, args.dataset, args.exp_name)
    args.max_acc = 0
    args.max_acc_mean = 0
    args.save_cnt = 0
    createDirectory(args.output)
    hyper_setting = ['true', str(args.mu_x_t) ,str(args.mu_y_t), str(args.t_std_major), str(args.t_std_ratio), 
    'false',str(args.mu_y_t) ,str(args.mu_y_f), str(args.f_std_major), str(args.f_std_ratio), 'original']
    wandb.init(project='project1', name=args.exp_name+'_'+"_".join(hyper_setting))
    wandb_args = {
            "backbone":'ResNet18',
            "batch_size": args.batch_size,
            'num_gpu': 'A6000*1'
            }
    wandb.config.update(wandb_args)   



    train_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.577, 0.4494, 0.4001],
                            std=[0.2628, 0.2395, 0.2383]),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomErasing(scale=(0.02, 0.3)) ]) # 0.25


    eval_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.577, 0.4494, 0.4001],
                            std=[0.2628, 0.2395, 0.2383])])
    train_dataset = NLA_Rafdb(args, phase='train', transform=train_transforms)
    test_dataset = NLA_Rafdb(args, phase='test', transform=eval_transforms)


    train_loader = torch.utils.data.DataLoader(train_dataset,
                                            batch_size=args.batch_size,
                                            shuffle=True,
                                            num_workers=args.workers,
                                            pin_memory=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=args.batch_size,
                                            shuffle=False,
                                            num_workers=args.workers,
                                            pin_memory=True)




    model = NLA_r18(args)
    device = torch.device('cuda:{}'.format(args.gpu))
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters() , lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)


    for idx, i in enumerate(range(1, args.epochs + 1)):
        train_acc, train_loss = train(args, idx, model, train_loader, optimizer, scheduler, device)
        wandb.log({"Train Acc@1": train_acc,
        "Train Loss" : train_loss,
        }, step=idx)

        print(f'Train: [{idx}/{args.epochs + 1}]\\t'
        f'Train Acc@1 {train_acc:.4f}\\t'
        f'Train Loss {train_loss:.3f}\\t')


        acc_metric, test_acc, test_loss = test(model, test_loader, device)
        classwise_acc, total_acc, top2_acc = acc_metric.final_score()
        wandb.log({"Test Acc@1": total_acc,
        "Test Loss" : test_loss,
        "Test Mean Acc" : np.mean(classwise_acc),
        "Top 2 Acc" : top2_acc,
        'Neutral' : classwise_acc[6],
        'Happiness' : classwise_acc[3],
        'Sadness' : classwise_acc[4],
        'Surprise' : classwise_acc[0],
        'Fear' : classwise_acc[1],
        'Disgust' : classwise_acc[2],
        'Anger' : classwise_acc[5],
        }, step=idx)

        print(f'Test: [{idx}/{args.epochs + 1}]\\t'
        f'Test Acc@1 {test_acc:.4f}\\t'
        f'Test Mean Acc {np.mean(classwise_acc):.4f}\\t'
        f'Test Loss {test_loss:.3f}\\t')
        print(f'class acc : {classwise_acc}')

        if args.max_acc_mean < np.mean(classwise_acc):
            save_classifier(model, 'best_mean', args)
            args.max_acc_mean = np.mean(classwise_acc)

        if args.max_acc < test_acc:
            save_classifier(model, 'best', args)
            args.max_acc = test_acc

        # if  idx % 5 == 0:
        #     save_classifier(model, str(idx), args)

if __name__ == '__main__':
    main()
""",

'third_party/nla/train.py': """\
import torch
from src.utils import *
from src.loss import *
from tqdm import tqdm
import torch.nn as nn



def train(args, idx, model, train_loader, optimizer, scheduler, device):
    running_loss = 0.0
    iter_cnt = 0
    correct_sum = 0
    acc = AccuraryLogger_top2(7)
    model.to(device)
    model.train()    
    # if 'GA' in args.exp_name:
    args.eps = exponential_scheduler(idx, args.epochs, args.slope, args.sch_bool)

    for image, labels, image2 in tqdm(train_loader):
        image = image.to(device)
        image2 = image2.to(device)
        labels = labels.to(device)

        output  = model(image)
        output2 = model(image2)
        # Loss
        cross_loss = nn.CrossEntropyLoss(reduction='none')(output, labels)
        t_major = args.t_std_major**2 + (args.t_std_major/args.t_std_ratio)**2
        t_minor = -args.t_std_major**2 + (args.t_std_major/args.t_std_ratio)**2

        f_major = args.f_std_major**2 + (args.f_std_major/args.f_std_ratio)**2
        f_minor = args.f_std_major**2 - (args.f_std_major/args.f_std_ratio)**2


        loss_naw = NAW(args, mu_x_t=args.mu_x_t, mu_y_t=args.mu_y_t, mu_x_f = args.mu_x_f, mu_y_f = args.mu_y_f, f_minor=f_minor, f_major=f_major, t_minor=t_minor, t_major=t_major, t_lambda=args.t_lambda)(output, labels)


        loss_jsd = args.lam_c * jensen_shannon_divergence(output, output2)

        loss = (args.lam_a * cross_loss.mean() + args.lam_b * loss_naw.mean()) + loss_jsd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        iter_cnt += 1
        _, predicts = torch.max(output, 1)

        preds = predicts.detach().cpu().numpy() 
        targets = labels.detach().cpu().numpy()

        preds_top2 = output.detach().cpu().numpy()
        acc.update(preds, preds_top2, targets)

        correct_num = torch.eq(predicts, labels).sum()
        correct_sum += correct_num
        running_loss += loss.item()

    scheduler.step()
    running_loss = running_loss / iter_cnt
    acc = correct_sum.float() / float(train_loader.dataset.__len__())
    return acc, running_loss
""",

'third_party/qcs/__init__.py': """\
# Local FER experiment package.
""",

'third_party/qcs/models/__init__.py': """\
# Local FER experiment package.
""",

'third_party/qcs/models/ir50.py': """\
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, Dropout2d, Dropout, AvgPool2d, \\
    MaxPool2d, AdaptiveAvgPool2d, Sequential, Module, Parameter
import torch.nn.functional as F
import torch
from collections import namedtuple
import math
import pdb


##################################  Original Arcface Model #############################################################

class Flatten(Module):
    def forward(self, input):
        return input.view(input.size(0), -1)


def l2_norm(input, axis=1):
    norm = torch.norm(input, 2, axis, True)
    output = torch.div(input, norm)
    return output


class SEModule(Module):
    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = AdaptiveAvgPool2d(1)
        self.fc1 = Conv2d(
            channels, channels // reduction, kernel_size=1, padding=0, bias=False)
        self.relu = ReLU(inplace=True)
        self.fc2 = Conv2d(
            channels // reduction, channels, kernel_size=1, padding=0, bias=False)
        self.sigmoid = Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x


# i = 0

class bottleneck_IR(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride, bias=False), BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3, 3), (1, 1), 1, bias=False), PReLU(depth),
            Conv2d(depth, depth, (3, 3), stride, 1, bias=False), BatchNorm2d(depth))
        i = 0

    def forward(self, x):
        shortcut = self.shortcut_layer(x)
        # print(shortcut.shape)
        # print('---s---')
        res = self.res_layer(x)
        # print(res.shape)
        # print('---r---')
        # i = i + 50
        # print(i)
        # print('50')
        return res + shortcut


class bottleneck_IR_SE(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR_SE, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride, bias=False),
                BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3, 3), (1, 1), 1, bias=False),
            PReLU(depth),
            Conv2d(depth, depth, (3, 3), stride, 1, bias=False),
            BatchNorm2d(depth),
            SEModule(depth, 16)
        )

    def forward(self, x):
        shortcut = self.shortcut_layer(x)
        res = self.res_layer(x)
        return res + shortcut


class Bottleneck(namedtuple('Block', ['in_channel', 'depth', 'stride'])):
    '''A named tuple describing a ResNet block.'''
    # print('50')


def get_block(in_channel, depth, num_units, stride=2):
    return [Bottleneck(in_channel, depth, stride)] + [Bottleneck(depth, depth, 1) for i in range(num_units - 1)]


def get_blocks(num_layers):
    if num_layers == 50:
        blocks1 = [
            get_block(in_channel=64, depth=64, num_units=3),
            # get_block(in_channel=64, depth=128, num_units=4),
            # get_block(in_channel=128, depth=256, num_units=14),
            # get_block(in_channel=256, depth=512, num_units=3)
        ]
        blocks2 = [
            # get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=4),
            # get_block(in_channel=128, depth=256, num_units=14),
            # get_block(in_channel=256, depth=512, num_units=3)
        ]
        blocks3 = [
            # get_block(in_channel=64, depth=64, num_units=3),
            # get_block(in_channel=64, depth=128, num_units=4),
            get_block(in_channel=128, depth=256, num_units=14),
            # get_block(in_channel=256, depth=512, num_units=3)
        ]

    elif num_layers == 100:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=13),
            get_block(in_channel=128, depth=256, num_units=30),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    elif num_layers == 152:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=8),
            get_block(in_channel=128, depth=256, num_units=36),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    return blocks1, blocks2, blocks3


class Backbone(Module):
    def __init__(self, num_layers, drop_ratio, mode='ir'):
        super(Backbone, self).__init__()
        # assert num_layers in [50, 100, 152], 'num_layers should be 50,100, or 152'
        assert mode in ['ir', 'ir_se'], 'mode should be ir or ir_se'
        blocks1, blocks2, blocks3 = get_blocks(num_layers)
        # blocks2 = get_blocks(num_layers)
        if mode == 'ir':
            unit_module = bottleneck_IR
        elif mode == 'ir_se':
            unit_module = bottleneck_IR_SE
        self.input_layer = Sequential(Conv2d(3, 64, (3, 3), 1, 1, bias=False),
                                      BatchNorm2d(64),
                                      PReLU(64))
        self.output_layer = Sequential(BatchNorm2d(512),
                                       Dropout(drop_ratio),
                                       Flatten(),
                                       Linear(512 * 7 * 7, 512),
                                       BatchNorm1d(512))
        modules1 = []
        for block in blocks1:
            for bottleneck in block:
                modules1.append(
                    unit_module(bottleneck.in_channel,
                                bottleneck.depth,
                                bottleneck.stride))

        modules2 = []
        for block in blocks2:
            for bottleneck in block:
                modules2.append(
                    unit_module(bottleneck.in_channel,
                                bottleneck.depth,
                                bottleneck.stride))

        modules3 = []
        for block in blocks3:
            for bottleneck in block:
                modules3.append(
                    unit_module(bottleneck.in_channel,
                                bottleneck.depth,
                                bottleneck.stride))
        # modules4 = []
        # for block in blocks4:
        #     for bottleneck in block:
        #         modules4.append(
        #             unit_module(bottleneck.in_channel,
        #                         bottleneck.depth,
        #                         bottleneck.stride))
        self.body1 = Sequential(*modules1)
        self.body2 = Sequential(*modules2)
        self.body3 = Sequential(*modules3)
        # self.body4 = Sequential(*modules4)

    def forward(self, x):
        x = F.interpolate(x, size=112)
        x = self.input_layer(x)
        x1 = self.body1(x)
        x2 = self.body2(x1)
        x3 = self.body3(x2)

        # x = self.output_layer(x)
        # return l2_norm(x)

        return x1, x2, x3

def load_pretrained_weights(model, checkpoint):
    import collections
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
    model_dict = model.state_dict()
    new_state_dict = collections.OrderedDict()
    matched_layers, discarded_layers = [], []
    for i, (k, v) in enumerate(state_dict.items()):
        # print(i)

        # If the pretrained state_dict was saved as nn.DataParallel,
        # keys would contain "module.", which should be ignored.
        if k.startswith('module.'):
            k = k[7:]
        if k in model_dict and model_dict[k].size() == v.size():

            new_state_dict[k] = v
            matched_layers.append(k)
            print(k)
        else:
            print(k)
            discarded_layers.append(k)
    # new_state_dict.requires_grad = False
    model_dict.update(new_state_dict)
    model.load_state_dict(model_dict)
    print('load_weight', len(matched_layers))
    return model
""",

'third_party/qcs/models/QCS_7cls_raf_db.py': """\
import torch
import torch.nn as nn
from torch.nn import functional as F
from .ir50 import Backbone
from .vit_model_7cls import VisionTransformer, PatchEmbed
from timm.models.layers import trunc_normal_, DropPath
from ..utils import *


def load_pretrained_weights(model, checkpoint):
    import collections
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
    model_dict = model.state_dict()
    new_state_dict = collections.OrderedDict()
    matched_layers, discarded_layers = [], []
    for k, v in state_dict.items():
        # If the pretrained state_dict was saved as nn.DataParallel,
        # keys would contain "module.", which should be ignored.
        if k.startswith('module.'):
            k = k[7:]
        if k.startswith('ir_back.'):
            k = k[8:]
        if k in model_dict and model_dict[k].size() == v.size():
            new_state_dict[k] = v
            matched_layers.append(k)
        else:
            discarded_layers.append(k)
    # new_state_dict.requires_grad = False
    model_dict.update(new_state_dict)

    model.load_state_dict(model_dict)
    model._pretrained_matched = matched_layers
    model._pretrained_discarded = discarded_layers

    print('load_weight', len(matched_layers))
    return model




class Mlp(nn.Module):
    \"\"\"
    MLP as used in Vision Transformer, MLP-Mixer and related networks
    \"\"\"
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x




class CrossAttention(nn.Module):

    def __init__(self, embed_dim=768):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.proj = nn.Linear(embed_dim, embed_dim * 2)  #

        self.mlp = Mlp(in_features=embed_dim, hidden_features=int(embed_dim * 4), act_layer=nn.GELU, drop=0)

        self.theta = nn.Parameter(torch.tensor(0.), requires_grad=True)

    def forward(self, x_a, x_p, x_n, x_n2):
        B, N, C = x_a.shape

        x_a = self.norm1(x_a)
        x_p = self.norm1(x_p)
        x_n = self.norm1(x_n)
        x_n2 = self.norm1(x_n2)

        qkv_a = self.proj(x_a).reshape(B, N, 2, C).permute(2, 0, 1, 3)
        QK1, V1 = qkv_a[0], qkv_a[1]
        # print(f1.shape, qkv1.shape)

        qkv_p = self.proj(x_p).reshape(B, N, 2, C).permute(2, 0, 1, 3)
        QK2, V2 = qkv_p[0], qkv_p[1]

        qkv_n = self.proj(x_n).reshape(B, N, 2, C).permute(2, 0, 1, 3)
        QK3, V3 = qkv_n[0], qkv_n[1]

        qkv_n2 = self.proj(x_n2).reshape(B, N, 2, C).permute(2, 0, 1, 3)
        QK4, V4 = qkv_n2[0], qkv_n2[1]

        ##############################################

        k = torch.tanh(self.theta)
        cross_map1, cross_map2, cross_map3, cross_map4 = Attn_QCS_SD(QK1, QK2, QK3, QK4, k) # # B WH WH # torch.Size([64, 49, 49])

        ############################################
        cross_map1 = torch.reshape(cross_map1, shape=(B, N, 1))
        attn_a = cross_map1 * V1  # B N C # torch.Size([64, 49, 768])
        x_a = x_a + attn_a

        cross_map2 = torch.reshape(cross_map2, shape=(B, N, 1))
        attn_p = cross_map2 * V2  # B N C
        x_p = x_p + attn_p

        cross_map3 = torch.reshape(cross_map3, shape=(B, N, 1))
        attn_n = cross_map3 * V3  # B N C
        x_n = x_n + attn_n

        cross_map4 = torch.reshape(cross_map4, shape=(B, N, 1))
        attn_n2 = cross_map4 * V4  # B N C
        x_n2 = x_n2 + attn_n2

        x_a = self.norm2(x_a)
        mlp_a = self.mlp(x_a)
        x_a = x_a + mlp_a

        x_p = self.norm2(x_p)
        mlp_p = self.mlp(x_p)
        x_p = x_p + mlp_p

        x_n = self.norm2(x_n)
        mlp_n = self.mlp(x_n)
        x_n = x_n + mlp_n

        x_n2 = self.norm2(x_n2)
        mlp_n2 = self.mlp(x_n2)
        x_n2 = x_n2 + mlp_n2

        return x_a, x_p, x_n, x_n2, k





class Fusion(nn.Module):

    def __init__(self, num_classes=7):
        super().__init__()

        self.bilinear_pooling = Bilinear_Pooling()
        self.classifier = nn.Linear(768*768, num_classes)

    def forward(self, x_a, x_p, x_n, x_n2, attn_a, attn_p, attn_n, attn_n2):

        pooling_a = torch.flatten(self.bilinear_pooling(x_a, attn_a), 1)
        out_a = self.classifier(pooling_a)

        pooling_p = torch.flatten(self.bilinear_pooling(x_p, attn_p), 1)
        out_p = self.classifier(pooling_p)

        pooling_n = torch.flatten(self.bilinear_pooling(x_n, attn_n), 1)
        out_n = self.classifier(pooling_n)

        pooling_n2 = torch.flatten(self.bilinear_pooling(x_n2, attn_n2), 1)
        out_n2 = self.classifier(pooling_n2)

        return out_a, out_p, out_n, out_n2





class pyramid_trans_expr(nn.Module):
    def __init__(self, img_size=224, num_classes=7, dims=[64, 128, 256], embed_dim=768, pretrained_path=None):
        super().__init__()

        self.img_size = img_size

        self.num_classes = num_classes

        self.VIT_base = VisionTransformer(depth=2, drop_ratio=0, embed_dim=embed_dim, num_classes=num_classes)
        self.VIT_cross = VisionTransformer(depth=1, drop_ratio=0, embed_dim=embed_dim, num_classes=num_classes)

        self.ir_back = Backbone(50, 0.0, 'ir')
        # Kaggle reload: a strict full-state load supplies the entire backbone.
        if pretrained_path is not None:
            ir_checkpoint = torch.load(pretrained_path, map_location='cpu', weights_only=True)
            self.ir_back = load_pretrained_weights(self.ir_back, ir_checkpoint)

        self.conv1 = nn.Conv2d(in_channels=dims[0], out_channels=dims[0], kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(in_channels=dims[1], out_channels=dims[1], kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(in_channels=dims[2], out_channels=dims[2], kernel_size=3, stride=2, padding=1)

        self.embed_q = nn.Sequential(nn.Conv2d(dims[0], 768, kernel_size=3, stride=2, padding=1),
                                     nn.Conv2d(768, 768, kernel_size=3, stride=2, padding=1))
        self.embed_k = nn.Sequential(nn.Conv2d(dims[1], 768, kernel_size=3, stride=2, padding=1))
        self.embed_v = PatchEmbed(img_size=14, patch_size=14, in_c=256, embed_dim=768)

        #It is recommended to replace the layer names (embed_q, embed_k, embed_v)
        # used in the old version of checkpoints with the new layer names (embed_1, embed_2, embed_3).
        #self.embed_1 = nn.Sequential(nn.Conv2d(dims[0], 768, kernel_size=3, stride=2, padding=1),
        #                             nn.Conv2d(768, 768, kernel_size=3, stride=2, padding=1))
        #self.embed_2 = nn.Sequential(nn.Conv2d(dims[1], 768, kernel_size=3, stride=2, padding=1))
        #self.embed_3 = PatchEmbed(img_size=14, patch_size=14, in_c=256, embed_dim=768)

        self.cross_attention_1 = CrossAttention(embed_dim)
        self.cross_attention_2 = CrossAttention(embed_dim)
        self.cross_attention_3 = CrossAttention(embed_dim)


        #self.fusion = Fusion(num_classes)


    def forward(self, x_a, x_p, x_n, x_n2):

        '''----------------- anchor ----------------'''
        x_a_ir1, x_a_ir2, x_a_ir3 = self.ir_back(x_a)
        x_a_ir1, x_a_ir2, x_a_ir3 = self.conv1(x_a_ir1), self.conv2(x_a_ir2), self.conv3(x_a_ir3)
        #print(x_ir1.shape, x_ir2.shape, x_ir3.shape)
        #torch.Size([64, 64, 28, 28]) torch.Size([64, 128, 14, 14]) torch.Size([64, 256, 7, 7])
        x_a_o1, x_a_o2, x_a_o3 = self.embed_q(x_a_ir1).flatten(2).transpose(1, 2), self.embed_k(x_a_ir2).flatten(2).transpose(1, 2), self.embed_v(x_a_ir3)
        #print(o1.shape, o2.shape, o3.shape)
        #torch.Size([64, 49, 768]) torch.Size([64, 49, 768]) torch.Size([64, 49, 768])
        x_a_o = torch.cat([x_a_o1, x_a_o2, x_a_o3], dim=1)
        x_a, x_a_0 = self.VIT_base(x_a_o)

        if x_p == None:
            return x_a

        '''----------------- positive ----------------'''
        x_p_ir1, x_p_ir2, x_p_ir3 = self.ir_back(x_p)
        x_p_ir1, x_p_ir2, x_p_ir3 = self.conv1(x_p_ir1), self.conv2(x_p_ir2), self.conv3(x_p_ir3)
        x_p_o1, x_p_o2, x_p_o3 = self.embed_q(x_p_ir1).flatten(2).transpose(1, 2), self.embed_k(x_p_ir2).flatten(2).transpose(1, 2), self.embed_v(x_p_ir3)
        x_p_o = torch.cat([x_p_o1, x_p_o2, x_p_o3], dim=1)
        x_p, x_p_0 = self.VIT_base(x_p_o)

        '''----------------- negative ----------------'''
        x_n_ir1, x_n_ir2, x_n_ir3 = self.ir_back(x_n)
        x_n_ir1, x_n_ir2, x_n_ir3 = self.conv1(x_n_ir1), self.conv2(x_n_ir2), self.conv3(x_n_ir3)
        x_n_o1, x_n_o2, x_n_o3 = self.embed_q(x_n_ir1).flatten(2).transpose(1, 2), self.embed_k(x_n_ir2).flatten(2).transpose(1, 2), self.embed_v(x_n_ir3)
        x_n_o = torch.cat([x_n_o1, x_n_o2, x_n_o3], dim=1)
        x_n, x_n_0 = self.VIT_base(x_n_o)

        '''----------------- negative2 ----------------'''
        x_n2_ir1, x_n2_ir2, x_n2_ir3 = self.ir_back(x_n2)
        x_n2_ir1, x_n2_ir2, x_n2_ir3 = self.conv1(x_n2_ir1), self.conv2(x_n2_ir2), self.conv3(x_n2_ir3)
        x_n2_o1, x_n2_o2, x_n2_o3 = self.embed_q(x_n2_ir1).flatten(2).transpose(1, 2), self.embed_k(x_n2_ir2).flatten(2).transpose(1, 2), self.embed_v(x_n2_ir3)
        x_n2_o = torch.cat([x_n2_o1, x_n2_o2, x_n2_o3], dim=1)
        x_n2, x_n2_0 = self.VIT_base(x_n2_o)


        '''----------------- attention ----------------'''
        _, N, _ = x_a_o1.shape
        x_a_0_1, x_a_0_2, x_a_0_3 = torch.split(x_a_0, [N, N, N], dim=1)
        x_p_0_1, x_p_0_2, x_p_0_3 = torch.split(x_p_0, [N, N, N], dim=1)
        x_n_0_1, x_n_0_2, x_n_0_3 = torch.split(x_n_0, [N, N, N], dim=1)
        x_n2_0_1, x_n2_0_2, x_n2_0_3 = torch.split(x_n2_0, [N, N, N], dim=1)

        attn_a1, attn_p1, attn_n_1, attn_n2_1, k1 = self.cross_attention_1(x_a_0_1, x_p_0_1, x_n_0_1, x_n2_0_1)
        attn_a2, attn_p2, attn_n_2, attn_n2_2, k2 = self.cross_attention_2(x_a_0_2, x_p_0_2, x_n_0_2, x_n2_0_2)
        attn_a3, attn_p3, attn_n_3, attn_n2_3, k3 = self.cross_attention_3(x_a_0_3, x_p_0_3, x_n_0_3, x_n2_0_3)

        attn_a_o = torch.cat([attn_a1, attn_a2, attn_a3], dim=1)
        attn_p_o = torch.cat([attn_p1, attn_p2, attn_p3], dim=1)
        attn_n_o = torch.cat([attn_n_1, attn_n_2, attn_n_3], dim=1)
        attn_n2_o = torch.cat([attn_n2_1, attn_n2_2, attn_n2_3], dim=1)

        '''----------------- connection ----------------'''
        x_a_0 = x_a_0 + attn_a_o
        x_p_0 = x_p_0 + attn_p_o
        x_n_0 = x_n_0 + attn_n_o
        x_n2_0 = x_n2_0 + attn_n2_o
        out_a, _ = self.VIT_cross(x_a_0)
        out_p, _ = self.VIT_cross(x_p_0)
        out_n, _ = self.VIT_cross(x_n_0)
        out_n2, _ = self.VIT_cross(x_n2_0)

        #out_a, out_p, out_n, out_n2 = self.fusion(x_a_0, x_p_0, x_n_0, x_n2_0, attn_a_o, attn_p_o, attn_n_o, attn_n2_o)


        return x_a, x_p, x_n, x_n2, out_a, out_p, out_n, out_n2



def compute_param_flop():
    from thop import profile
    model = pyramid_trans_expr()
    img = torch.rand(size=(1,3,224,224))
    flops, params = profile(model, inputs=(img,))
    print(f'flops:{flops/1000**3}G,params:{params/1000**2}M')
""",

'third_party/qcs/models/vit_model_7cls.py': """\
\"\"\"

\"\"\"
from functools import partial
from collections import OrderedDict

import torch
import torch.nn as nn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.hub
from functools import partial
# import mat
# from vision_transformer.ir50 import Backbone


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.hub
from functools import partial
import math

from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model
from timm.models.vision_transformer import _cfg, Mlp, Block
# from .ir50 import Backbone


def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    \"\"\"3x3 convolution with padding\"\"\"
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    \"\"\"1x1 convolution\"\"\"
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


def drop_path(x, drop_prob: float = 0., training: bool = False):
    \"\"\"
    Drop paths (Stochastic Depth) per sample (when applied in main path of residual blocks).
    This is the same as the DropConnect impl I created for EfficientNet, etc networks, however,
    the original name is misleading as 'Drop Connect' is a different form of dropout in a separate paper...
    I've opted for changing the layer and argument names to 'drop path' rather than mix DropConnect as a layer name and use
    'survival rate' as the argument.
    \"\"\"
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # work with diff dim tensors, not just 2D ConvNets
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()  # binarize
    output = x.div(keep_prob) * random_tensor
    return output


class BasicBlock(nn.Module):
    __constants__ = ['downsample']

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        norm_layer = nn.BatchNorm2d
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class DropPath(nn.Module):
    \"\"\"
    Drop paths (Stochastic Depth) per sample  (when applied in main path of residual blocks).
    \"\"\"

    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class PatchEmbed(nn.Module):
    \"\"\"
    2D Image to Patch Embedding
    \"\"\"

    def __init__(self, img_size=14, patch_size=16, in_c=256, embed_dim=768, norm_layer=None):
        super().__init__()
        img_size = (img_size, img_size)
        patch_size = (patch_size, patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]

        self.proj = nn.Conv2d(256, 768, kernel_size=1)
        self.norm = norm_layer(embed_dim) if norm_layer else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        # assert H == self.img_size[0] and W == self.img_size[1], \\
        #     f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."

        # flatten: [B, C, H, W] -> [B, C, HW]
        # transpose: [B, C, HW] -> [B, HW, C]
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x


class Attention(nn.Module):
    def __init__(self,
                 dim, in_chans,  # 输入token的dim
                 num_heads=8,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop_ratio=0.,
                 proj_drop_ratio=0.):
        super(Attention, self).__init__()
        self.num_heads = 8
        self.img_chanel = in_chans + 1
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop_ratio)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop_ratio)

    def forward(self, x):
        x_img = x[:, :self.img_chanel, :]
        B, N, C = x_img.shape

        qkv = self.qkv(x_img).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x_img = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x_img = self.proj(x_img)
        x_img = self.proj_drop(x_img)

        return x_img


class AttentionBlock(nn.Module):
    __constants__ = ['downsample']

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(AttentionBlock, self).__init__()
        norm_layer = nn.BatchNorm2d
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride
        # self.cbam = CBAM(planes, 16)
        #self.inplanes = inplanes
        self.eca_block = eca_block()

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        #inplanes = self.inplanes
        out = self.eca_block(out)
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Mlp(nn.Module):
    \"\"\"
    MLP as used in Vision Transformer, MLP-Mixer and related networks
    \"\"\"

    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Block(nn.Module):
    def __init__(self,
                 dim, in_chans,
                 num_heads,
                 mlp_ratio=4.,
                 qkv_bias=False,
                 qk_scale=None,
                 drop_ratio=0.,
                 attn_drop_ratio=0.,
                 drop_path_ratio=0.,
                 act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm):
        super(Block, self).__init__()
        self.norm1 = norm_layer(dim)
        self.img_chanel = in_chans + 1

        self.conv = nn.Conv1d(self.img_chanel, self.img_chanel, 1)
        self.attn = Attention(dim, in_chans=in_chans, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
                              attn_drop_ratio=attn_drop_ratio, proj_drop_ratio=attn_drop_ratio)#proj_drop_ratio=drop_ratio
        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path_ratio) if drop_path_ratio > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop_ratio)

    def forward(self, x):
        # x = x + self.drop_path(self.attn(self.norm1(x)))
        # x = x + self.drop_path(self.mlp(self.norm2(x)))

        x_img = x
        x_img = x_img + self.drop_path(self.attn(self.norm1(x)))
        x = x_img + self.drop_path(self.mlp(self.norm2(x_img)))

        return x


class ClassificationHead(nn.Module):
    def __init__(self, input_dim: int, target_dim: int):
        super().__init__()
        self.linear = torch.nn.Linear(input_dim, target_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        y_hat = self.linear(x)
        return y_hat


def load_pretrained_weights(model, checkpoint):
    import collections
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint
    model_dict = model.state_dict()
    new_state_dict = collections.OrderedDict()
    matched_layers, discarded_layers = [], []
    for k, v in state_dict.items():
        # If the pretrained state_dict was saved as nn.DataParallel,
        # keys would contain "module.", which should be ignored.
        if k.startswith('module.'):
            k = k[7:]
        if k in model_dict and model_dict[k].size() == v.size():
            new_state_dict[k] = v
            matched_layers.append(k)
        else:
            discarded_layers.append(k)
    # new_state_dict.requires_grad = False
    model_dict.update(new_state_dict)

    model.load_state_dict(model_dict)
    print('load_weight', len(matched_layers))
    return model

class eca_block(nn.Module):
    def __init__(self, channel=128, b=1, gamma=2):
        super(eca_block, self).__init__()
        kernel_size = int(abs((math.log(channel, 2) + b) / gamma))
        kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)



class eca_block(nn.Module):
    def __init__(self, channel=196, b=1, gamma=2):
        super(eca_block, self).__init__()
        kernel_size = int(abs((math.log(channel, 2) + b) / gamma))
        kernel_size = kernel_size if kernel_size % 2 else kernel_size + 1

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

class SE_block(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.linear1 = torch.nn.Linear(input_dim, input_dim)
        self.relu = nn.ReLU()
        self.linear2 = torch.nn.Linear(input_dim, input_dim)
        self.sigmod = nn.Sigmoid()

    def forward(self, x):
        x1 = self.linear1(x)
        x1 = self.relu(x1)
        x1 = self.linear2(x1)
        x1 = self.sigmod(x1)
        x = x * x1
        return x


class VisionTransformer(nn.Module):
    def __init__(self, img_size=14, patch_size=14, in_c=147, num_classes=7,
                 embed_dim=768, depth=6, num_heads=8, mlp_ratio=4.0, qkv_bias=True,
                 qk_scale=None, representation_size=None, distilled=False, drop_ratio=0.,
                 attn_drop_ratio=0., drop_path_ratio=0., embed_layer=PatchEmbed, norm_layer=None,
                 act_layer=None):
        \"\"\"
        Args:
            img_size (int, tuple): input image size
            patch_size (int, tuple): patch size
            in_c (int): number of input channels
            num_classes (int): number of classes for classification head
            embed_dim (int): embedding dimension
            depth (int): depth of transformer
            num_heads (int): number of attention heads
            mlp_ratio (int): ratio of mlp hidden dim to embedding dim
            qkv_bias (bool): enable bias for qkv if True
            qk_scale (float): override default qk scale of head_dim ** -0.5 if set
            representation_size (Optional[int]): enable and set representation layer (pre-logits) to this value if set
            distilled (bool): model includes a distillation token and head as in DeiT models
            drop_ratio (float): dropout rate
            attn_drop_ratio (float): attention dropout rate
            drop_path_ratio (float): stochastic depth rate
            embed_layer (nn.Module): patch embedding layer
            norm_layer: (nn.Module): normalization layer
        \"\"\"
        super(VisionTransformer, self).__init__()
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim  # num_features for consistency with other models
        self.num_tokens = 2 if distilled else 1
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        act_layer = act_layer or nn.GELU

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, in_c + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_ratio)

        self.se_block = SE_block(input_dim=embed_dim)


        self.patch_embed = embed_layer(img_size=img_size, patch_size=patch_size, in_c=256, embed_dim=768)
        num_patches = self.patch_embed.num_patches
        self.head = ClassificationHead(input_dim=embed_dim, target_dim=self.num_classes)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        # self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_ratio)
        # self.IR = IR()
        self.eca_block = eca_block()



        self.CON1 = nn.Conv2d(256, 768, kernel_size=1, stride=1, bias=False)
        self.IRLinear1 = nn.Linear(1024, 768)
        self.IRLinear2 = nn.Linear(768, 512)
        self.eca_block = eca_block()
        dpr = [x.item() for x in torch.linspace(0, drop_path_ratio, depth)]  # stochastic depth decay rule
        self.blocks = nn.Sequential(*[
            Block(dim=embed_dim, in_chans=in_c, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                  qk_scale=qk_scale,
                  drop_ratio=drop_ratio, attn_drop_ratio=attn_drop_ratio, drop_path_ratio=dpr[i],
                  norm_layer=norm_layer, act_layer=act_layer)
            for i in range(depth)
        ])
        self.norm = norm_layer(embed_dim)

        # Representation layer
        if representation_size and not distilled:
            self.has_logits = True
            self.num_features = representation_size
            self.pre_logits = nn.Sequential(OrderedDict([
                ("fc", nn.Linear(embed_dim, representation_size)),
                ("act", nn.Tanh())
            ]))
            print('pre_logits')
        else:
            self.has_logits = False
            self.pre_logits = nn.Identity()

        # Classifier head(s)
        # self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.head_dist = None
        if distilled:
            self.head_dist = nn.Linear(self.embed_dim, self.num_classes) if num_classes > 0 else nn.Identity()

        # Weight init
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        if self.dist_token is not None:
            nn.init.trunc_normal_(self.dist_token, std=0.02)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(_init_vit_weights)

    def forward_features(self, x):
        # [B, C, H, W] -> [B, num_patches, embed_dim]
        # x = self.patch_embed(x)  # [B, 196, 768]
        # [1, 1, 768] -> [B, 1, 768]


        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        if self.dist_token is None:
            x = torch.cat((cls_token, x), dim=1)  # [B, 147, 768]
        else:
            x = torch.cat((cls_token, self.dist_token.expand(x.shape[0], -1, -1), x), dim=1)

        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x) # attention
        x = self.norm(x)


        if self.dist_token is None:
            #torch.Size([B, 147, 768])
            return self.pre_logits(x[:, 0]), x[:, 1:x.shape[1]]
        else:
            return x[:, 0], x[:, 1]

    def forward(self, x):  # x[B,147,768]

        x, x1 = self.forward_features(x)
        x = self.se_block(x)
        # x[B,768]   x1[B,7]
        x = self.head(x)


        return x, x1


def _init_vit_weights(m):
    \"\"\"
    ViT weight initialization
    :param m: module
    \"\"\"
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)
""",

'third_party/qcs/sam.py': """\
import torch


class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)

        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)  # climb to the local maximum "w + e(w)"

        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  # get back to "w" from "w + e(w)"

        self.base_optimizer.step()  # do the actual "sharpness-aware" update

        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "Sharpness Aware Minimization requires closure, but it was not provided"
        closure = torch.enable_grad()(closure)  # the closure should do a full forward-backward pass

        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device  # put everything on the same device, in case of model parallelism
        norm = torch.norm(
                    torch.stack([
                        ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups
""",

'third_party/qcs/utils.py': """\
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import shutil
import math






class KLDiv(nn.Module):
    \"\"\"Distilling the Knowledge in a Neural Network\"\"\"
    def __init__(self, T):
        super(KLDiv, self).__init__()
        self.T = T

    def forward(self, y_s, y_t):
        p_s = F.log_softmax(y_s/self.T, dim=1)
        p_t = F.softmax(y_t/self.T, dim=1)
        loss = F.kl_div(p_s, p_t, reduction='batchmean') * (self.T**2)
        return loss




class Bilinear_Pooling(nn.Module):
    def __init__(self,  **kwargs):
        super(Bilinear_Pooling, self).__init__()

    def forward(self, feature_map1, feature_map2):
        feature_map1 = feature_map1.permute(0, 2, 1)
        feature_map2 = feature_map2.permute(0, 2, 1)

        B, C, N1 = feature_map1.size()  # torch.Size([64, 768, 147])
        B, C, N2 = feature_map2.size()

        X = torch.bmm(feature_map1, feature_map2.transpose(1, 2)) / N1
        # torch.Size([64, 768*768])
        X = torch.reshape(X, (B, C * C))
        X = torch.sign(X) * torch.sqrt(torch.abs(X) + 1e-5)
        bilinear_features = 100 * torch.nn.functional.normalize(X)
        return bilinear_features




class EuclideanLoss(torch.nn.Module):
    def __init__(self):
        super(EuclideanLoss, self).__init__()

    def forward(self, x, y):
        B, _ = x.shape
        #margin = 3.0

        dist = torch.sqrt(((x - y) ** 2).sum())
        dist_loss = (dist / B)

        #euclidean_distance = torch.nn.functional.pairwise_distance(x, y)
        #dist_loss = torch.mean(torch.pow(torch.clamp(margin - euclidean_distance, min=0.0), 2))

        return dist_loss






'''-----------------------  SDPA  ------------------------'''
def Attn_HW(f_K1, f_Q2):
    #B, 3*WH, C = feature1.shape  # torch.Size([64, 147, 768])
    B, N, C = f_K1.shape
    scale = C ** -0.5

    I = (f_Q2 @ f_K1.transpose(-2, -1)) * scale   #（N2,N1）
    I2 = F.softmax(I, 2)
    I1 = F.softmax(I, 1).permute(0, 2, 1)

    return I1, I2







'''-----------------------  DCS  S  ------------------------'''
def Attn_DCS_S(f_K1, f_Q2):
    #B, 3*WH, C = feature1.shape  # torch.Size([64, 147, 768])
    B, N, C = f_K1.shape

    D = torch.cdist(f_Q2, f_K1, p=2)  # B x N2 x C , B x N1 x C --> B x N2 x N1
    D = D - torch.min(D)
    S = torch.max(D) - D

    S1 = F.normalize(S, p=2, dim=2)
    S1 = torch.sum(S1, dim=1)  # B N1
    cross_map1 = F.softmax(S1, dim=-1)

    S2 = F.normalize(S, p=2, dim=1)
    S2 = torch.sum(S2, dim=2)   #B N2
    cross_map2 = F.softmax(S2, dim=-1)


    return cross_map1, cross_map2





'''-----------------------  QCS  SD  ------------------------'''
def Attn_QCS_SD(QK1, QK2, QK3, QK4, k):
    B, N, C = QK1.shape
    k = k + 1

    D_2x1 = torch.cdist(QK2, QK1, p=2)  # B x N2 x C , B x N1 x C --> B x N2 x N1
    D_2x1 = D_2x1 - torch.min(D_2x1)
    S_2x1 = torch.max(D_2x1) - D_2x1

    D_4x3 = torch.cdist(QK4, QK3, p=2)  # B x N4 x C , B x N3 x C --> B x N4 x N3
    D_4x3 = D_4x3 - torch.min(D_4x3)
    S_4x3 = torch.max(D_4x3) - D_4x3

    D_3x1 = torch.cdist(QK3, QK1, p=2)  # B x N3 x N1
    D_4x2 = torch.cdist(QK4, QK2, p=2)  # B x N4 x N2
    D_3x1 = D_3x1 - torch.min(D_3x1)
    D_4x2 = D_4x2 - torch.min(D_4x2)

    ######################## K1 ########################
    S_2x1_n1 = F.normalize(S_2x1, p=2, dim=2)
    S1 = torch.sum(S_2x1_n1, dim=1)  # B x N1
    ''''''''''''''''''''''''''
    D_3x1_n1 = F.normalize(D_3x1, p=2, dim=2)
    D1 = torch.sum(D_3x1_n1, dim=1)  # B x N1

    ''''''''''''''''''''''''''
    SD1 = S1 + k * D1
    cross_map1 = F.softmax(SD1, dim=-1)

    ######################## Q4 ########################
    S_4x3_n4 = F.normalize(S_4x3, p=2, dim=1)
    S4 = torch.sum(S_4x3_n4, dim=2)  # B x N4
    ''''''''''''''''''''''''''
    D_4x2_n4 = F.normalize(D_4x2, p=2, dim=1)
    D4 = torch.sum(D_4x2_n4, dim=2)  # B x N4

    ''''''''''''''''''''''''''
    SD4 = S4 + k * D4
    cross_map4 = F.softmax(SD4, dim=-1)

    ######################## K2 ########################
    S_1x2 = S_2x1.transpose(1, 2)  # B x N1 x N2
    S_1x2_n2 = F.normalize(S_1x2, p=2, dim=2)
    S2 = torch.sum(S_1x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    D_4x2_n2 = F.normalize(D_4x2, p=2, dim=2)
    D2 = torch.sum(D_4x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    SD2 = S2 + k * D2
    cross_map2 = F.softmax(SD2, dim=-1)

    ######################## Q3 ########################
    S_3x4 = S_4x3.transpose(1, 2)  # B x N3 x N4
    S_3x4_n3 = F.normalize(S_3x4, p=2, dim=1)
    S3 = torch.sum(S_3x4_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    D_3x1_n3 = F.normalize(D_3x1, p=2, dim=1)
    D3 = torch.sum(D_3x1_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    SD3 = S3 + k * D3
    cross_map3 = F.softmax(SD3, dim=-1)

    return cross_map1, cross_map2, cross_map3, cross_map4



'''-----------------------   QCS S  ------------------------'''
def Attn_QCS_S(QK1, QK2, QK3, QK4, k):
    B, N, C = QK1.shape
    #k = k + 1

    D_2x1 = torch.cdist(QK2, QK1, p=2)  # B x N2 x C , B x N1 x C --> B x N2 x N1
    D_2x1 = D_2x1 - torch.min(D_2x1)
    S_2x1 = torch.max(D_2x1) - D_2x1

    D_4x3 = torch.cdist(QK4, QK3, p=2)  # B x N4 x C , B x N3 x C --> B x N4 x N3
    D_4x3 = D_4x3 - torch.min(D_4x3)
    S_4x3 = torch.max(D_4x3) - D_4x3

    #D_3x1 = torch.cdist(QK3, QK1, p=2)  # B x N3 x N1
    #D_4x2 = torch.cdist(QK4, QK2, p=2)  # B x N4 x N2
    #D_3x1 = D_3x1 - torch.min(D_3x1)
    #D_4x2 = D_4x2 - torch.min(D_4x2)

    ######################## K1 ########################
    S_2x1_n1 = F.normalize(S_2x1, p=2, dim=2)
    S1 = torch.sum(S_2x1_n1, dim=1)  # B x N1
    ''''''''''''''''''''''''''
    #D_3x1_n1 = F.normalize(D_3x1, p=2, dim=2)
    #D1 = torch.sum(D_3x1_n1, dim=1)  # B x N1

    ''''''''''''''''''''''''''
    #SD1 = S1 + k * D1
    cross_map1 = F.softmax(S1, dim=-1)

    ######################## Q4 ########################
    S_4x3_n4 = F.normalize(S_4x3, p=2, dim=1)
    S4 = torch.sum(S_4x3_n4, dim=2)  # B x N4
    ''''''''''''''''''''''''''
    #D_4x2_n4 = F.normalize(D_4x2, p=2, dim=1)
    #D4 = torch.sum(D_4x2_n4, dim=2)  # B x N4

    ''''''''''''''''''''''''''
    #SD4 = S4 + k * D4
    cross_map4 = F.softmax(S4, dim=-1)

    ######################## K2 ########################
    S_1x2 = S_2x1.transpose(1, 2)  # B x N1 x N2
    S_1x2_n2 = F.normalize(S_1x2, p=2, dim=2)
    S2 = torch.sum(S_1x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    #D_4x2_n2 = F.normalize(D_4x2, p=2, dim=2)
    #D2 = torch.sum(D_4x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    #SD2 = S2 + k * D2
    cross_map2 = F.softmax(S2, dim=-1)

    ######################## Q3 ########################
    S_3x4 = S_4x3.transpose(1, 2)  # B x N3 x N4
    S_3x4_n3 = F.normalize(S_3x4, p=2, dim=1)
    S3 = torch.sum(S_3x4_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    #D_3x1_n3 = F.normalize(D_3x1, p=2, dim=1)
    #D3 = torch.sum(D_3x1_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    #SD3 = S3 + k * D3
    cross_map3 = F.softmax(S3, dim=-1)

    return cross_map1, cross_map2, cross_map3, cross_map4




'''----------------------- QCS D  ------------------------'''
def Attn_QCS_D(QK1, QK2, QK3, QK4, k):
    B, N, C = QK1.shape
    #k = k + 1

    #D_2x1 = torch.cdist(QK2, QK1, p=2)  # B x N2 x C , B x N1 x C --> B x N2 x N1
    #D_2x1 = D_2x1 - torch.min(D_2x1)
    #S_2x1 = torch.max(D_2x1) - D_2x1

    #D_4x3 = torch.cdist(QK4, QK3, p=2)  # B x N4 x C , B x N3 x C --> B x N4 x N3
    #D_4x3 = D_4x3 - torch.min(D_4x3)
    #S_4x3 = torch.max(D_4x3) - D_4x3

    D_3x1 = torch.cdist(QK3, QK1, p=2)  # B x N3 x N1
    D_4x2 = torch.cdist(QK4, QK2, p=2)  # B x N4 x N2
    D_3x1 = D_3x1 - torch.min(D_3x1)
    D_4x2 = D_4x2 - torch.min(D_4x2)

    ######################## K1 ########################
    #S_2x1_n1 = F.normalize(S_2x1, p=2, dim=2)
    #S1 = torch.sum(S_2x1_n1, dim=1)  # B x N1
    ''''''''''''''''''''''''''
    D_3x1_n1 = F.normalize(D_3x1, p=2, dim=2)
    D1 = torch.sum(D_3x1_n1, dim=1)  # B x N1

    ''''''''''''''''''''''''''
    #SD1 = S1 + k * D1
    cross_map1 = F.softmax(D1, dim=-1)

    ######################## Q4 ########################
    #S_4x3_n4 = F.normalize(S_4x3, p=2, dim=1)
    #S4 = torch.sum(S_4x3_n4, dim=2)  # B x N4
    ''''''''''''''''''''''''''
    D_4x2_n4 = F.normalize(D_4x2, p=2, dim=1)
    D4 = torch.sum(D_4x2_n4, dim=2)  # B x N4

    ''''''''''''''''''''''''''
    #SD4 = S4 + k * D4
    cross_map4 = F.softmax(D4, dim=-1)

    ######################## K2 ########################
    #S_1x2 = S_2x1.transpose(1, 2)  # B x N1 x N2
    #S_1x2_n2 = F.normalize(S_1x2, p=2, dim=2)
    #S2 = torch.sum(S_1x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    D_4x2_n2 = F.normalize(D_4x2, p=2, dim=2)
    D2 = torch.sum(D_4x2_n2, dim=1)  # B x N2
    ''''''''''''''''''''''''''
    #SD2 = S2 + k * D2
    cross_map2 = F.softmax(D2, dim=-1)

    ######################## Q3 ########################
    #S_3x4 = S_4x3.transpose(1, 2)  # B x N3 x N4
    #S_3x4_n3 = F.normalize(S_3x4, p=2, dim=1)
    #S3 = torch.sum(S_3x4_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    D_3x1_n3 = F.normalize(D_3x1, p=2, dim=1)
    D3 = torch.sum(D_3x1_n3, dim=2)  # B x N3
    ''''''''''''''''''''''''''
    #SD3 = S3 + k * D3
    cross_map3 = F.softmax(D3, dim=-1)

    return cross_map1, cross_map2, cross_map3, cross_map4
""",

'third_party/SOURCES.json': """\
{
  "retrieved": "2026-10-11",
  "files": [
    {
      "path": "third_party/emonext/models.py",
      "repository": "yelboudouri/EmoNeXt",
      "commit": "ce77dc13e13d3b09742f0f4efb9dd83fcf3b6c95",
      "upstream_path": "models.py",
      "upstream_blob_sha": "6c645324a957f81f98ccc2dabde828674d3e8601",
      "source_url": "https://github.com/yelboudouri/EmoNeXt/blob/ce77dc13e13d3b09742f0f4efb9dd83fcf3b6c95/models.py"
    },
    {
      "path": "third_party/nla/loss.py",
      "repository": "jungyu0413/NLA",
      "commit": "7bc363b559699c28afd1a19f4bc8b2857f917c40",
      "upstream_path": "src/loss.py",
      "upstream_blob_sha": "ed92d05c11bcc964c059d14fe5b44242ffc18689",
      "source_url": "https://github.com/jungyu0413/NLA/blob/7bc363b559699c28afd1a19f4bc8b2857f917c40/src/loss.py"
    },
    {
      "path": "third_party/nla/train.py",
      "repository": "jungyu0413/NLA",
      "commit": "7bc363b559699c28afd1a19f4bc8b2857f917c40",
      "upstream_path": "src/train.py",
      "upstream_blob_sha": "833e0260c2730b267a1e87aaea161c93a310aece",
      "source_url": "https://github.com/jungyu0413/NLA/blob/7bc363b559699c28afd1a19f4bc8b2857f917c40/src/train.py"
    },
    {
      "path": "third_party/nla/main_rafdb.py",
      "repository": "jungyu0413/NLA",
      "commit": "7bc363b559699c28afd1a19f4bc8b2857f917c40",
      "upstream_path": "main_rafdb.py",
      "upstream_blob_sha": "706a5c8d2895f6e91050ec64b0c29956f13cd6b2",
      "source_url": "https://github.com/jungyu0413/NLA/blob/7bc363b559699c28afd1a19f4bc8b2857f917c40/main_rafdb.py"
    },
    {
      "path": "third_party/qcs/models/QCS_7cls_raf_db.py",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "models/QCS_7cls_raf_db.py",
      "upstream_blob_sha": "746aa7db097de9fb2c588fb42ae9bb0ecb524e59",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/models/QCS_7cls_raf_db.py"
    },
    {
      "path": "third_party/qcs/models/vit_model_7cls.py",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "models/vit_model_7cls.py",
      "upstream_blob_sha": "62b7ab3405921655611c6f962d901d3207920f74",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/models/vit_model_7cls.py"
    },
    {
      "path": "third_party/qcs/models/ir50.py",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "models/ir50.py",
      "upstream_blob_sha": "ed8e4bfcde77c3abae359bb20d0fbbacdf7973bb",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/models/ir50.py"
    },
    {
      "path": "third_party/qcs/utils.py",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "utils.py",
      "upstream_blob_sha": "3ebf777ce53281bdd33b471b1c3efb532d267ea4",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/utils.py"
    },
    {
      "path": "third_party/qcs/sam.py",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "data_processing/sam.py",
      "upstream_blob_sha": "d82f0bda0e59e17700306311ccf5ad578edb4b39",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/data_processing/sam.py"
    },
    {
      "path": "third_party/qcs/LICENSE",
      "repository": "birdwcp/QCS",
      "commit": "293f28f948dccdca385917fbc0cfeafd6220d2e2",
      "upstream_path": "LICENSE",
      "upstream_blob_sha": "4edaf82e3b2386d3b50e9de7850fc615508b1e10",
      "source_url": "https://github.com/birdwcp/QCS/blob/293f28f948dccdca385917fbc0cfeafd6220d2e2/LICENSE"
    }
  ],
  "note": "QCS local adaptation is documented in experiments/README.md; other references are retained for traceability."
}
""",

'third_party/qcs/LICENSE': """\
MIT License

Copyright (c) 2024 QCS

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
""",
})


## 3. 数据、训练、损失与选模代码

原训练集 28,275 张；去除 1,123 个同标签像素副本，以及 39 个冲突标签组的 110 张图像。
保留 27,042 张，按类别分层划为训练 21,633 / 验证 5,409，固定种子 `20261011`。
训练/验证像素 SHA-256 交集为零。内部类别顺序为
`Angry, Fear, Happy, Neutral, Sad, Surprise`。

预处理严格沿用原项目：48×48 灰度像素除以 255，双线性缩放，复制三通道，ImageNet 标准化。
ConvNeXt/QCS/EmoNeXt 输入 224×224，EmotiEff-B2 输入 260×260，并使用 max pooling。
训练模块保留 AdamW、EMA、SAM、Mixup、EAC 和固定验证集的替换组件选模流程。


In [ ]:
write_sources({
'fer/__init__.py': """\
# Local FER experiment package.
""",

'fer/data.py': """\
import json
import math
from pathlib import Path
import random

import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import Dataset


class FERDataset(Dataset):
    def __init__(self, directory, split_name, quadruplets=False):
        self.directory = Path(directory)
        self.split = json.loads((self.directory/'split.json').read_text(encoding='utf-8'))
        self.rows = [json.loads(l) for l in (self.directory/'manifest.jsonl').read_text(encoding='utf-8').splitlines()]
        self.indices = self.split[split_name]
        self.images = np.load(self.directory/'images.npy', mmap_mode='r')
        self.labels = np.array([self.rows[i]['label'] for i in self.indices])
        self.quadruplets = quadruplets
        self.by_class = {int(c):np.where(self.labels == c)[0].tolist() for c in np.unique(self.labels)}
        if quadruplets and split_name != 'train':
            raise ValueError('Quadruplet partners must come exclusively from the training split.')

    def __len__(self):
        return len(self.indices)

    def get_image(self, position):
        return torch.from_numpy(self.images[self.indices[position]].copy())

    def __getitem__(self, position):
        label = int(self.labels[position])
        if not self.quadruplets:
            return self.get_image(position), label, self.indices[position]
        positive = random.choice(self.by_class[label])
        while positive == position:
            positive = random.choice(self.by_class[label])
        negative_label = random.choice([c for c in self.by_class if c != label])
        negative, negative2 = random.sample(self.by_class[negative_label], 2)
        return (torch.stack([self.get_image(i) for i in (position, positive, negative, negative2)]),
                label, negative_label)


class ImagePipeline:
    \"\"\"Per-image GPU augmentation; defaults preserve the first comparison.\"\"\"
    def __init__(self, device, size=224, augmentation=None):
        self.size = size
        self.augmentation = augmentation or {}
        self.mean = torch.tensor([.485,.456,.406], device=device).view(1,3,1,1)
        self.std = torch.tensor([.229,.224,.225], device=device).view(1,3,1,1)

    @torch.no_grad()
    def __call__(self, raw, training=False):
        x = raw.unsqueeze(1).float().div_(255.)
        batch = x.shape[0]
        if training:
            degrees = self.augmentation.get('degrees',10.)
            scale_range = self.augmentation.get('scale_range',.05)
            translation = self.augmentation.get('translation',.03)
            jitter = self.augmentation.get('jitter',.08)
            angle = (torch.rand(batch, device=x.device)*2*degrees-degrees) * math.pi / 180.
            scale = torch.rand(batch, device=x.device)*2*scale_range + 1-scale_range
            flip = torch.where(torch.rand(batch, device=x.device)<.5, -1., 1.)
            theta = torch.zeros(batch,2,3,device=x.device)
            theta[:,0,0] = torch.cos(angle)*scale*flip
            theta[:,0,1] = -torch.sin(angle)*scale
            theta[:,1,0] = torch.sin(angle)*scale*flip
            theta[:,1,1] = torch.cos(angle)*scale
            theta[:,:,2] = (torch.rand(batch,2,device=x.device)-.5)*2*translation
            grid = F.affine_grid(theta, [batch,1,self.size,self.size], align_corners=False)
            x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
            brightness = torch.rand(batch,1,1,1,device=x.device)*2*jitter + 1-jitter
            contrast = torch.rand(batch,1,1,1,device=x.device)*2*jitter + 1-jitter
            mean = x.mean((2,3), keepdim=True)
            x = ((x-mean)*contrast+mean).mul_(brightness).clamp_(0,1)
            erase_prob = self.augmentation.get('erase_probability',0.)
            if erase_prob:
                apply = torch.rand(batch,1,1,1,device=x.device)<erase_prob
                area = torch.rand(batch,1,1,1,device=x.device)*.06+.02
                ratio = torch.rand(batch,1,1,1,device=x.device)*1.5+.5
                height = (area*ratio).sqrt()*self.size
                width = (area/ratio).sqrt()*self.size
                top = torch.rand(batch,1,1,1,device=x.device)*(self.size-height)
                left = torch.rand(batch,1,1,1,device=x.device)*(self.size-width)
                yy = torch.arange(self.size,device=x.device).view(1,1,-1,1)
                xx = torch.arange(self.size,device=x.device).view(1,1,1,-1)
                mask = apply & (yy>=top) & (yy<top+height) & (xx>=left) & (xx<left+width)
                x = torch.where(mask,x.mean((2,3),keepdim=True),x)
        else:
            x = F.interpolate(x, size=(self.size,self.size), mode='bilinear', align_corners=False)
        return ((x.expand(-1,3,-1,-1)-self.mean)/self.std).contiguous(memory_format=torch.channels_last)
""",

'fer/emotieff.py': """\
\"\"\"EmotiEffLib transfer with explicit pooling and six-class head mapping.

Official source: sb-ai-lab/EmotiEffLib, commit
520a051c64cd191521e5934655314e769a319684. No upstream code is executed.
\"\"\"
import torch
from torch import nn
from torch.nn import functional as F
import timm


TARGET_CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
SOURCE_CLASSES = {
    7: ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise'],
    8: ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise'],
}
CLASS_ROWS = {7: [0, 2, 3, 4, 5, 6], 8: [0, 3, 4, 5, 6, 7]}
VARIANTS = {
    'b0': dict(architecture='tf_efficientnet_b0', pool='avg', image_size=224, source_classes=8),
    'b2': dict(architecture='tf_efficientnet_b2', pool='max', image_size=260, source_classes=7),
}


def map_classifier(state, source_classes):
    \"\"\"Keep matching source logits, rather than randomly replacing the head.\"\"\"
    if source_classes not in CLASS_ROWS:
        raise ValueError('Only the documented AffectNet seven/eight-class mappings are supported.')
    result = dict(state)
    for suffix in ('weight', 'bias'):
        source = result.pop('classifier.0.' + suffix, None)
        if source is None:
            source = result['classifier.' + suffix]
        if source.shape[0] != source_classes:
            raise ValueError('Classifier shape disagrees with the source class order.')
        result['classifier.' + suffix] = source[CLASS_ROWS[source_classes]].clone()
    return result


class FEREmotiEff(nn.Module):
    def __init__(self, variant='b0', dropout=0.):
        super().__init__()
        spec = VARIANTS[variant]
        self.backbone = timm.create_model(spec['architecture'], pretrained=False,
                                          num_classes=6, global_pool=spec['pool'],
                                          drop_rate=dropout, drop_path_rate=0.)

    def forward(self, x, return_aux=False, return_attention=False, attention_only=False):
        features = self.backbone.forward_features(x)
        if return_attention or attention_only:
            # EAC's public implementation detaches the classifier weights and
            # compares all unnormalised class maps. Gradients reach both views.
            with torch.autocast(device_type=x.device.type, enabled=False):
                attention = F.conv2d(features.float(),
                                     self.backbone.classifier.weight.detach().float()[:, :, None, None])
            if attention_only:
                return attention
        logits = self.backbone.forward_head(features)
        if return_attention:
            return logits, attention
        return (logits, logits.new_zeros(())) if return_aux else logits
""",

'fer/io.py': """\
\"\"\"Atomic artifacts that tolerate temporary Windows file-sharing locks.\"\"\"
from contextlib import contextmanager
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys
import time
from uuid import uuid4


def _retry_file_access(operation, *, timeout):
    deadline = time.monotonic() + timeout
    delay = 0.025
    while True:
        try:
            return operation()
        except OSError as error:
            temporary_lock = isinstance(error, PermissionError) or getattr(error, 'winerror', None) in (5, 32, 33)
            remaining = deadline - time.monotonic()
            if not temporary_lock or remaining <= 0:
                raise
            time.sleep(min(delay, remaining))
            delay = min(delay * 2, 0.25)


def replace_with_retry(source, destination, *, timeout=5.0):
    return _retry_file_access(lambda: os.replace(source, destination), timeout=timeout)


def read_json(path, *, timeout=5.0):
    return _retry_file_access(lambda: json.loads(Path(path).read_text(encoding='utf-8')),
                              timeout=timeout)


@contextmanager
def atomic_output(path, *, timeout=5.0):
    path = Path(path)
    # Each writer gets its own file on the destination filesystem. Close the
    # creation handle before another library opens it (required on Windows).
    temp = path.with_name(f'{path.name}.{uuid4().hex}.tmp')
    try:
        with temp.open('xb'):
            pass
        yield temp
        replace_with_retry(temp, path, timeout=timeout)
    finally:
        try:
            temp.unlink(missing_ok=True)
        except OSError as error:
            print(f'Warning: could not remove temporary artifact {temp}: {error}', file=sys.stderr, flush=True)


def atomic_json(path, value, *, required=True, timeout=5.0):
    payload = json.dumps(value, ensure_ascii=False, indent=2) + '\\n'
    try:
        with atomic_output(path, timeout=timeout) as temp:
            temp.write_text(payload, encoding='utf-8')
        return True
    except OSError as error:
        if required:
            raise
        print(f'Warning: could not update status {path}; retaining previous status: {error}',
              file=sys.stderr, flush=True)
        return False


def update_status(path, value, *, required=False, timeout=5.0):
    return atomic_json(path, {**value, 'updated_at': datetime.now(timezone.utc).isoformat()},
                       required=required, timeout=timeout)
""",

'fer/losses.py': """\
\"\"\"NLA loss adaptation, preserving the authors' weighting and consistency formula.

Reference: third_party/nla/loss.py and train.py. This is a loss transfer onto
EmoNeXt, not a reproduction of the original ResNet18/MS-Celeb-1M experiment.
\"\"\"
import math

import torch
from torch.nn import functional as F


def eac_consistency(attention, flipped_attention):
    \"\"\"Exact horizontal alignment, equivalent to EAC's integer flip grid.\"\"\"
    if attention.shape != flipped_attention.shape or attention.ndim != 4:
        raise ValueError('EAC requires matching N,C,H,W class activation maps.')
    return F.mse_loss(attention.float(), flipped_attention.float().flip(-1))


def gaussian_weight(points, mean, diagonal, off_diagonal):
    covariance = points.new_tensor([[diagonal, off_diagonal], [off_diagonal, diagonal]])
    difference = points - points.new_tensor(mean)[:,None]
    exponent = -.5 * (difference * (torch.linalg.inv(covariance) @ difference)).sum(0)
    return torch.exp(exponent) / (2*math.pi*torch.sqrt(torch.det(covariance)))


def nla_loss(logits, flipped_logits, labels, epoch_index, epochs, reference_batch=32):
    # Compute covariance and divergences in FP32 under mixed precision.
    logits, flipped_logits = logits.float(), flipped_logits.float()
    probabilities = logits.softmax(1)
    ground_truth = probabilities.gather(1, labels[:,None]).squeeze(1)
    top_two = probabilities.topk(2, dim=1).values
    correct = top_two[:,0].eq(ground_truth)
    nearest_negative = torch.where(correct, top_two[:,1], top_two[:,0])
    points = torch.stack([ground_truth, nearest_negative])
    schedule = 1-math.exp(-15*epoch_index/max(epochs,1))
    true_major = .8**2 + (.8/2)**2
    true_minor = (-.8**2 + (.8/2)**2)*schedule
    false_major = .91**2 + (.91/6)**2
    false_minor = .91**2 - (.91/6)**2
    true_weight = gaussian_weight(points, [.5,.5], true_major, true_minor)
    false_weight = gaussian_weight(points, [.30,.15], false_major, false_minor)
    weights = torch.where(correct, true_weight, false_weight)
    ce = F.cross_entropy(logits, labels, reduction='none')
    # The public implementation uses a logits-mean target and reverse KL.
    # Preserve that behavior; do not silently substitute the standard JS formula.
    middle = ((logits+flipped_logits)*.5).softmax(1)
    consistency = .5*(F.kl_div(logits.log_softmax(1), middle, reduction='none').sum(1)
                       + F.kl_div(flipped_logits.log_softmax(1), middle, reduction='none').sum(1))
    # The original loss sums over its batch of 32. Make this scale explicit
    # so gradient accumulation does not alter its relative contribution.
    return .5*ce.mean()+.5*(ce*weights).mean()+.5*reference_batch*consistency.mean()
""",

'fer/models.py': """\
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

from third_party.emonext.models import EmoNeXt


class FERConvNeXt(EmoNeXt):
    def __init__(self, variant='emonext', num_classes=6, backbone='tiny', drop_path_rate=.1):
        if backbone not in ('tiny', 'small'):
            raise ValueError(f'Unknown ConvNeXt backbone: {backbone}')
        super().__init__(num_classes=num_classes, drop_path_rate=drop_path_rate,
                         depths=[3,3,27,3] if backbone=='small' else [3,3,9,3])
        self.variant = variant
        if variant == 'convnext':
            self.localization = nn.Identity()
            self.fc_loc = nn.Identity()
            self.attention = nn.Identity()
            for layer in self.downsample_layers[1:]:
                layer[2] = nn.Identity()

    def stn(self, x):
        if self.variant == 'convnext':
            return x
        # flatten supports channels-last tensors; upstream view assumes contiguous NCHW.
        theta = self.fc_loc(self.localization(x).flatten(1)).reshape(-1,2,3)
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        return F.grid_sample(x, grid, align_corners=True)

    def forward(self, x, return_aux=False):
        features = self.forward_features(self.stn(x))
        logits = self.head(features)
        if not return_aux:
            return logits
        regularizer = logits.new_zeros(())
        if self.variant != 'convnext':
            _, weights = self.attention(features)
            regularizer = (weights - weights.mean()).square().mean()
        return logits, regularizer


def make_model(config, load_pretrained=True):
    if config['model'] == 'emotieff':
        from .emotieff import FEREmotiEff, TARGET_CLASSES, VARIANTS
        variant = config['backbone']
        if config.get('image_size',224) != VARIANTS[variant]['image_size']:
            raise ValueError('Use the official EmotiEff input resolution for this comparison.')
        model = FEREmotiEff(variant, dropout=config.get('dropout',0.))
        info = dict(pretraining='EmotiEffLib: face identification then AffectNet expression',
                    backbone=variant, classes=TARGET_CLASSES, pool=VARIANTS[variant]['pool'])
        if load_pretrained:
            checkpoint = torch.load(config['pretrained'],map_location='cpu',weights_only=True)
            if checkpoint['metadata']['classes'] != TARGET_CLASSES or checkpoint['metadata']['variant'] != variant:
                raise RuntimeError('Converted pretrained label mapping or architecture mismatch.')
            model.backbone.load_state_dict(checkpoint['model'],strict=True)
            info.update(checkpoint['metadata'])
        return model, info
    if config['model'] == 'qcs':
        from third_party.qcs.models.QCS_7cls_raf_db import pyramid_trans_expr
        model = pyramid_trans_expr(num_classes=6, pretrained_path=config['pretrained'] if load_pretrained else None)
        # Unused face-identification output layer is never called by upstream forward.
        model.ir_back.output_layer = nn.Identity()
        unmatched = [key for key in model.ir_back.state_dict()
                     if load_pretrained and key not in model.ir_back._pretrained_matched and not key.endswith('num_batches_tracked')]
        if unmatched:
            raise RuntimeError(f'IR50 backbone weights not fully initialized: {unmatched[:10]}')
        return model, {'pretraining':'MS-Celeb-1M IR50', 'classes':6,
                       'loaded_tensors':len(getattr(model.ir_back, '_pretrained_matched', []))}
    backbone = config.get('backbone','tiny')
    model = FERConvNeXt(variant=config['model'], num_classes=6, backbone=backbone,
                       drop_path_rate=config.get('drop_path_rate',.1))
    info = {'pretraining':f'ImageNet-22K ConvNeXt-{backbone.title()}', 'classes':6}
    if load_pretrained:
        checkpoint = torch.load(Path(config['pretrained']), map_location='cpu', weights_only=True)
        state = checkpoint['model']
        state = {k:v for k,v in state.items() if not k.startswith('head.')}
        result = model.load_state_dict(state, strict=False)
        expected = ('head.', 'localization.', 'fc_loc.', 'attention.')
        unexpected_missing = [k for k in result.missing_keys
                              if not k.startswith(expected) and '.2.fc.' not in k]
        if result.unexpected_keys or unexpected_missing:
            raise RuntimeError(f'Unexpected pretrained mismatch: {result.unexpected_keys}/{unexpected_missing}')
        info.update(loaded_tensors=len(state), missing_task_tensors=result.missing_keys)
    return model, info
""",

'fer/prepare.py': """\
\"\"\"Create a reproducible, exact-duplicate-free FER development split.\"\"\"
import argparse
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import hashlib
import json
from pathlib import Path
import time

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

CLASSES = ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


def inspect(item):
    root, path, label, split = item
    with Image.open(path) as image:
        image.load()
        if image.size != (48, 48) or image.mode != 'L':
            raise ValueError(f'Unexpected image format: {path}: {image.size}/{image.mode}')
        pixels = np.array(image, dtype=np.uint8)
    digest = hashlib.sha256(pixels.tobytes()).hexdigest()
    return dict(path=path.relative_to(root).as_posix(), label=label, original_split=split,
                pixel_sha256=digest, constant=bool(pixels.min() == pixels.max())), pixels


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--root', type=Path, default=Path('.'))
    parser.add_argument('--output', type=Path, default=Path('artifacts/data'))
    parser.add_argument('--seed', type=int, default=20261011)
    parser.add_argument('--val-fraction', type=float, default=.2)
    args = parser.parse_args()
    root = args.root.resolve()
    out = args.output.resolve()
    out.mkdir(parents=True, exist_ok=True)
    if (out / 'split.json').exists():
        raise FileExistsError('A locked split already exists; use a new output directory to regenerate.')
    jobs = []
    for label, name in enumerate(CLASSES):
        jobs.extend((root, p, label, 'train') for p in sorted((root/'train'/name).glob('*.jpg')))
    jobs.extend((root, p, -1, 'test') for p in sorted((root/'test').glob('*.jpg')))
    images = np.lib.format.open_memmap(out/'images.npy', mode='w+', dtype=np.uint8,
                                     shape=(len(jobs), 48, 48))
    rows = []
    start = time.time()
    with ThreadPoolExecutor(max_workers=12) as pool:
        for i, (row, pixels) in enumerate(pool.map(inspect, jobs)):
            row['index'] = i
            rows.append(row)
            images[i] = pixels
            if (i+1) % 5000 == 0:
                print(f'Scanned {i+1}/{len(jobs)} images ({time.time()-start:.1f}s)', flush=True)
    images.flush()
    groups = defaultdict(list)
    for row in rows:
        if row['original_split'] == 'train':
            groups[row['pixel_sha256']].append(row)
    clean, excluded = [], []
    conflict_groups = 0
    for digest, members in sorted(groups.items()):
        labels = {r['label'] for r in members}
        if len(labels) > 1:
            conflict_groups += 1
            reason = 'conflicting_labels_for_identical_pixels'
        elif members[0]['constant']:
            reason = 'constant_image'
        else:
            clean.append(members[0]['index'])
            excluded.extend(dict(index=r['index'], path=r['path'], reason='same_label_duplicate',
                                 representative=members[0]['path']) for r in members[1:])
            continue
        excluded.extend(dict(index=r['index'], path=r['path'], reason=reason) for r in members)
    train, val = train_test_split(clean, test_size=args.val_fraction, random_state=args.seed,
                                 stratify=[rows[i]['label'] for i in clean])
    train, val = sorted(train), sorted(val)
    train_hashes = {rows[i]['pixel_sha256'] for i in train}
    val_hashes = {rows[i]['pixel_sha256'] for i in val}
    assert not train_hashes & val_hashes
    assert len(train_hashes) == len(train) and len(val_hashes) == len(val)
    test = [r['index'] for r in rows if r['original_split'] == 'test']
    manifest = ''.join(json.dumps(r, ensure_ascii=False, sort_keys=True)+'\\n' for r in rows)
    (out/'manifest.jsonl').write_bytes(manifest.encode('utf-8'))
    split = dict(version=1, seed=args.seed, classes=CLASSES, train=train, val=val, test=test,
                 manifest_sha256=hashlib.sha256(manifest.encode('utf-8')).hexdigest(),
                 selection_policy='one representative per nonconstant, label-consistent pixel hash; stratified 80/20',
                 group_key='sha256 of decoded 48x48 grayscale pixels')
    (out/'split.json').write_text(json.dumps(split, ensure_ascii=False, indent=2), encoding='utf-8')
    (out/'excluded.json').write_text(json.dumps(excluded, ensure_ascii=False, indent=2), encoding='utf-8')
    all_train_hashes = set(groups)
    summary = dict(original_train=sum(r['original_split']=='train' for r in rows),
                   original_test=len(test), clean_unique=len(clean), train=len(train), val=len(val),
                   train_class_counts={CLASSES[k]:v for k,v in sorted(Counter(rows[i]['label'] for i in train).items())},
                   val_class_counts={CLASSES[k]:v for k,v in sorted(Counter(rows[i]['label'] for i in val).items())},
                   exclusions=dict(Counter(r['reason'] for r in excluded)),
                   conflicting_pixel_groups=conflict_groups, train_val_pixel_overlap=0,
                   test_images_matching_original_train=sum(rows[i]['pixel_sha256'] in all_train_hashes for i in test),
                   manifest_sha256=split['manifest_sha256'], seconds=round(time.time()-start,2))
    (out/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(summary, ensure_ascii=True, indent=2), flush=True)


if __name__ == '__main__':
    main()
""",

'fer/refinement.py': """\
\"\"\"Validation-only component replacement scoring and explicit module freezing.\"\"\"
import hashlib
from pathlib import Path

import numpy as np

from .io import read_json


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda: file.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def freeze_modules(model, config):
    names = list(config.get('freeze_modules', []))
    for stage in range(config.get('freeze_stages', 0)):
        names.extend([f'downsample_layers.{stage}', f'stages.{stage}'])
    modules = {name: model.get_submodule(name) for name in dict.fromkeys(names)}
    for module in modules.values():
        module.requires_grad_(False)
        module.eval()
    return modules


class ReplacementReference:
    \"\"\"Fixed other-component predictions; never participates in gradient training.\"\"\"

    def __init__(self, spec, indices, labels, classes, manifest_sha256):
        self.spec = spec
        self.indices = np.asarray(indices)
        self.labels = np.asarray(labels)
        self.classes = list(classes)
        self.weight = float(spec['candidate_weight'])
        self.fixed = np.zeros((len(labels), len(classes)), dtype=np.float32)
        if file_sha256(spec['replaces_checkpoint']) != spec['replaces_checkpoint_sha256']:
            raise ValueError('Replacement parent checkpoint hash mismatch.')
        if not 0 < self.weight < 1:
            raise ValueError('Candidate weight must be between zero and one.')
        if not np.isclose(self.weight + sum(c['weight'] for c in spec['components']), 1.):
            raise ValueError('Replacement weights must sum to one.')
        paths = [str(Path(c['checkpoint']).resolve()) for c in spec['components']]
        if len(paths) != len(set(paths)):
            raise ValueError('Reference checkpoints must be distinct.')
        if str(Path(spec['replaces_checkpoint']).resolve()) in paths:
            raise ValueError('Replaced model must not appear among fixed components.')
        for component in spec['components']:
            if component['weight'] <= 0:
                raise ValueError('Reference weights must be positive.')
            for field in ('checkpoint', 'probabilities', 'metadata'):
                if file_sha256(component[field]) != component[field + '_sha256']:
                    raise ValueError(f'Reference {field} hash mismatch.')
            metadata = read_json(component['metadata'])
            if (metadata['checkpoint_sha256'] != component['checkpoint_sha256'] or
                    metadata['manifest_sha256'] != manifest_sha256 or
                    metadata['flip_tta'] or metadata['eval_precision'] != 'float32' or
                    metadata['eval_batch_size'] != 32):
                raise ValueError('Reference prediction provenance mismatch.')
            if spec['classes'] != self.classes or spec['manifest_sha256'] != manifest_sha256:
                raise ValueError('Reference class order or manifest mismatch.')
            with np.load(component['probabilities'], allow_pickle=False) as saved:
                self.check_alignment(saved['labels'], saved['indices'])
                probs = saved['probabilities']
                self.check_probabilities(probs)
                self.fixed += component['weight'] * probs

    def check_alignment(self, labels, indices):
        if not np.array_equal(indices, self.indices) or not np.array_equal(labels, self.labels):
            raise ValueError('Reference validation indices or labels are misaligned.')

    def check_probabilities(self, probs):
        if (probs.shape != self.fixed.shape or not np.isfinite(probs).all() or
                np.any(probs < 0) or not np.allclose(probs.sum(1), 1., atol=1e-6, rtol=0)):
            raise ValueError('Invalid replacement probabilities.')

    def combine(self, labels, probabilities, indices):
        self.check_alignment(labels, indices)
        self.check_probabilities(probabilities)
        return self.fixed + self.weight * probabilities
""",

'fer/selection.py': """\
\"\"\"A small predefined recipe search over held-out validation probabilities.\"\"\"
from itertools import product
import numpy as np
from sklearn.metrics import f1_score


def recipe_candidates(probabilities, labels, profiles, qcs_weights=(.25,.5,.75), reference_keys=(),
                      new_keys=(), reference_blend_weights=(), replacement_parents=None):
    candidates=[]
    seen=set()

    def add(name, components):
        signature=tuple(sorted((key,round(weight,12)) for key,weight in components))
        if signature in seen:
            return
        seen.add(signature)
        if not np.isclose(sum(weight for _,weight in components),1.):
            raise ValueError('Recipe weights must sum to one.')
        combined=sum(probabilities[key]*weight for key,weight in components)
        if combined.shape!=(len(labels),6) or not np.isfinite(combined).all():
            raise ValueError('Invalid candidate probabilities.')
        prediction=combined.argmax(1)
        candidates.append(dict(name=name,components=[dict(profile=key,weight=weight) for key,weight in components],
                               accuracy=float(np.mean(prediction==labels)),
                               macro_f1=float(f1_score(labels,prediction,labels=range(6),average='macro',zero_division=0))))

    for key in probabilities:
        add(key,[(key,1.)])
    singles=sorted(candidates,key=lambda r:(r['accuracy'],r['macro_f1']),reverse=True)
    # At most one inference variant of each trained model enters an ensemble.
    ranked=[]
    runs=set()
    for row in singles:
        key=row['components'][0]['profile']
        run=profiles[key]['run']
        same_predictions=any(np.array_equal(probabilities[key],probabilities[previous]) for previous in ranked)
        if run not in runs and not same_predictions:
            ranked.append(key)
            runs.add(run)
    for n in (2,3):
        if len(ranked)>=n:
            add(f'mean_top_{n}',[(key,1./n) for key in ranked[:n]])
    if reference_keys:
        add('reference_mean',[(key,1./len(reference_keys)) for key in reference_keys])
        for key in new_keys:
            if key in reference_keys:
                raise ValueError('New blend candidates must be separate from the reference ensemble.')
            for weight in reference_blend_weights:
                if not 0 < weight < 1:
                    raise ValueError('Reference blend weights must lie strictly between zero and one.')
                add(f'reference_plus_{key}_{weight:.2f}',
                    [(ref,(1-weight)/len(reference_keys)) for ref in reference_keys]+[(key,weight)])
    if replacement_parents:
        choices={key:[key] for key in reference_keys}
        for child,parent in replacement_parents.items():
            if parent not in choices or child not in new_keys or child not in probabilities or child in reference_keys:
                raise ValueError('Replacement children and parents must belong to new and reference models.')
            choices[parent].append(child)
        if np.prod([len(values) for values in choices.values()]) > 32:
            raise ValueError('Replacement search exceeds the bounded recipe budget.')
        for keys in product(*choices.values()):
            changed=[key for key in keys if key not in reference_keys]
            if changed:
                add('replace_'+'__'.join(changed),[(key,1./len(keys)) for key in keys])
    qcs=next((key for key in ranked if profiles[key]['model']=='qcs'),None)
    other=next((key for key in ranked if profiles[key]['model']!='qcs'),None)
    if qcs is not None and other is not None:
        for weight in qcs_weights:
            add(f'best_non_qcs_plus_qcs_{weight:.2f}',[(other,1-weight),(qcs,weight)])
    return sorted(candidates,key=lambda r:(r['accuracy'],r['macro_f1'],-len(r['components'])),reverse=True)
""",

'fer/verify.py': """\
\"\"\"Integration checks: split integrity, pretrained loading, loss parity, GPU steps.\"\"\"
import gc
import argparse
import hashlib
import json
from pathlib import Path
import time
from types import SimpleNamespace

import numpy as np
import torch
from torch.nn import functional as F

from .data import FERDataset, ImagePipeline
from .losses import nla_loss
from .models import make_model
from .train import seed_everything, build_optimizer, forward_loss, check_gradient_finite, atomic_json


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--only', choices=['qcs','all'], default='all')
    parser.add_argument('--qcs-batch', type=int, default=8)
    args_cli=parser.parse_args()
    seed_everything(42)
    directory=Path('artifacts/data')
    train=FERDataset(directory,'train')
    val=FERDataset(directory,'val')
    assert hashlib.sha256((directory/'manifest.jsonl').read_bytes()).hexdigest()==train.split['manifest_sha256']
    assert set(train.indices).isdisjoint(val.indices)
    assert {train.rows[i]['pixel_sha256'] for i in train.indices}.isdisjoint(
        {val.rows[i]['pixel_sha256'] for i in val.indices})
    assert set(train.labels)==set(range(6)) and set(val.labels)==set(range(6))
    from third_party.nla.loss import NAW, jensen_shannon_divergence
    logits=torch.randn(7,6,device='cuda',requires_grad=True)
    flipped=torch.randn(7,6,device='cuda',requires_grad=True)
    targets=torch.tensor([0,1,2,3,4,5,0],device='cuda')
    epoch,epochs=2,12
    args=SimpleNamespace(eps=1-np.exp(-15*epoch/epochs))
    original=NAW(args,.5,.5,.3,.15,
                 f_minor=.91**2-(.91/6)**2,f_major=.91**2+(.91/6)**2,
                 t_minor=-.8**2+(.8/2)**2,t_major=.8**2+(.8/2)**2)
    reference=.5*F.cross_entropy(logits,targets)+.5*original(logits,targets).mean()
    reference+=.5*jensen_shannon_divergence(logits,flipped)*32/len(targets)
    adapted=nla_loss(logits,flipped,targets,epoch,epochs)
    torch.testing.assert_close(adapted,reference,rtol=2e-5,atol=2e-5)
    adapted.backward()
    assert torch.isfinite(logits.grad).all()
    pipeline=ImagePipeline('cuda')
    results=[]
    for name,kind,loss_name,batch in [('convnext_tiny','convnext','ce',16),
                                     ('emonext_tiny','emonext','ce',16),
                                     ('emonext_nla','emonext','nla',16),
                                     ('qcs','qcs','ce',args_cli.qcs_batch)]:
        if args_cli.only=='qcs' and kind!='qcs':
            continue
        seed_everything(42)
        config=dict(name=name,model=kind,loss=loss_name,epochs=12,lr=1e-4,
                    effective_batch_size=32,label_smoothing=.2,weight_decay=1e-4,
                    pretrained='artifacts/weights/'+('ir50_msceleb.pth' if kind=='qcs' else 'convnext_tiny_22k_224.pth'))
        model,info=make_model(config)
        model=model.to(device='cuda',memory_format=torch.channels_last)
        optimizer=build_optimizer(model,config)
        labels=torch.tensor([i%6 for i in range(batch)],device='cuda')
        raw=torch.stack([train[i][0] for i in range(batch)]).to('cuda')
        images=pipeline(raw,training=True)
        negative=(labels+1)%6
        input_images=[images,images.flip(-1),images,images.flip(-1)] if kind=='qcs' else images
        torch.cuda.reset_peak_memory_stats()
        timings=[]
        for step in range(4):
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            started=time.perf_counter()
            with torch.autocast('cuda',dtype=torch.bfloat16):
                loss,logits=forward_loss(model,input_images,labels,config,0,negative)
            assert logits.shape==(batch,6) and torch.isfinite(loss)
            loss.backward()
            if step==0: check_gradient_finite(model)
            if kind=='qcs':
                optimizer.first_step(zero_grad=True)
                with torch.autocast('cuda',dtype=torch.bfloat16):
                    second,_=forward_loss(model,input_images,labels,config,0,negative)
                second.backward()
                optimizer.second_step(zero_grad=True)
            else:
                optimizer.step()
            torch.cuda.synchronize()
            timings.append(time.perf_counter()-started)
        result=dict(name=name,batch_size=batch,pretrained=info,parameters=sum(p.numel() for p in model.parameters()),
                    seconds_per_step=float(np.median(timings[1:])),last_loss=loss.item(),
                    peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/2**30,
                    integration_status='passed')
        print(json.dumps(result),flush=True)
        results.append(result)
        atomic_json('artifacts/integration_checks'+('_qcs' if args_cli.only=='qcs' else '')+'.json',dict(split_overlap=0,nla_loss_parity='passed',runs=results))
        del model,optimizer,images,input_images,loss,logits
        if kind=='qcs': del second
        gc.collect()
        torch.cuda.empty_cache()
    atomic_json('artifacts/integration_checks'+('_qcs' if args_cli.only=='qcs' else '')+'.json',dict(split_overlap=0,nla_loss_parity='passed',runs=results))
    print('All integration checks passed.',flush=True)


if __name__=='__main__':
    main()
""",

'fer/verify_refine.py': """\
\"\"\"Check exact initialization, frozen state and real accumulated GPU updates.\"\"\"
import gc
import time

import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset

from .data import FERDataset, ImagePipeline
from .io import atomic_json, read_json
from .models import make_model
from .refinement import ReplacementReference, file_sha256, freeze_modules
from .train import EMA, build_optimizer, check_gradient_finite, evaluate, forward_loss, mixup_batch, seed_everything


def main():
    plan=read_json('experiments/refine_plan.json')
    configs=[read_json(path) for path in plan['configs']]
    ce,eac=configs[:2]
    differences={key for key in ce.keys()|eac.keys() if ce.get(key)!=eac.get(key)}
    assert differences=={'name','output','loss','eac_weight'},differences
    training,validation=(FERDataset(plan['data'],split) for split in ('train','val'))
    assert set(training.indices).isdisjoint(validation.indices)
    assert {training.rows[i]['pixel_sha256'] for i in training.indices}.isdisjoint(
        {validation.rows[i]['pixel_sha256'] for i in validation.indices})
    manifest=file_sha256('artifacts/data/manifest.jsonl')
    assert manifest==training.split['manifest_sha256']
    results=[]
    for config in configs:
        started=time.perf_counter()
        print(f'Checking {config["name"]}: load checkpoint and reproduce full validation baseline.',flush=True)
        seed_everything(config['seed'])
        checkpoint=torch.load(config['finetune_from'],map_location='cpu',weights_only=True)
        assert checkpoint['classes']==validation.split['classes']
        assert checkpoint['config']['manifest_sha256']==manifest
        model,_=make_model(config,load_pretrained=False)
        model.load_state_dict(checkpoint['model'],strict=True)
        del checkpoint
        frozen=freeze_modules(model,config)
        model=model.to(device='cuda',memory_format=torch.channels_last)
        optimizer=build_optimizer(model,config)
        ema=EMA(model)
        pipeline=ImagePipeline('cuda',size=config['image_size'],augmentation=config['augmentation'])
        scorer=ReplacementReference(config['selection_reference'],validation.indices,validation.labels,
                                     validation.split['classes'],manifest)
        labels,probs,indices=evaluate(model,DataLoader(validation,batch_size=32,num_workers=0,pin_memory=True),
                                       pipeline,config['model'],'cuda',precision='float32')
        fused=scorer.combine(labels,probs,indices)
        initial_accuracy=float(np.mean(fused.argmax(1)==labels))
        assert initial_accuracy==config['selection_reference']['baseline_accuracy'],initial_accuracy
        print(f'Initial fusion {initial_accuracy:.8%}; testing accumulated gradients and EMA.',flush=True)
        rng=np.random.default_rng(config['seed'])
        positions=rng.choice(len(training),size=config['batch_size']*6,replace=False)
        assert len(np.unique(training.labels[positions]))==6
        loader=DataLoader(Subset(training,positions.tolist()),batch_size=config['batch_size'])
        model.train()
        for module in frozen.values():
            module.eval()
        frozen_before={name:{key:value.detach().cpu().clone() for key,value in module.state_dict().items()}
                       for name,module in frozen.items()}
        head=model.backbone.classifier if config['model']=='emotieff' else model.head
        before=head.weight.detach().clone()
        torch.cuda.reset_peak_memory_stats()
        losses,timings,loss_parts=[],[],[]
        optimizer.zero_grad(set_to_none=True)
        for batch_index,(raw,labels,_) in enumerate(loader):
            tick=time.perf_counter()
            images=pipeline(raw.to('cuda'),training=True)
            labels=labels.to('cuda')
            images,targets=mixup_batch(images,labels,config.get('mixup_alpha',0.),config.get('mixup_probability',0.))
            with torch.autocast('cuda',dtype=torch.bfloat16):
                loss,logits=forward_loss(model,images,targets,config,0)
            assert torch.isfinite(loss) and logits.shape==(len(labels),6)
            ce_loss=F.cross_entropy(logits.float(),targets,label_smoothing=config['label_smoothing'])
            loss_parts.append(dict(ce=float(ce_loss.detach()),weighted_regularizer=float((loss-ce_loss).detach())))
            (loss/config['accumulation_steps']).backward()
            if (batch_index+1)%config['accumulation_steps']==0:
                check_gradient_finite(model)
                torch.nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)
            torch.cuda.synchronize()
            losses.append(float(loss.detach()))
            timings.append(time.perf_counter()-tick)
        assert not torch.equal(before,head.weight)
        assert ema.updates==3
        for name,module in frozen.items():
            for key,value in module.state_dict().items():
                torch.testing.assert_close(value.cpu(),frozen_before[name][key],rtol=0,atol=0)
            assert all(parameter.grad is None for parameter in module.parameters())
        check_labels,check_probs,check_indices=evaluate(ema.model,
                DataLoader(Subset(validation,range(32)),batch_size=32),pipeline,config['model'],'cuda',precision='float32')
        assert np.isfinite(check_probs).all()
        np.testing.assert_allclose(check_probs.sum(1),1.,atol=1e-6)
        np.testing.assert_array_equal(check_indices,validation.indices[:32])
        result=dict(name=config['name'],status='passed',initial_ensemble_accuracy=initial_accuracy,
                    losses=losses,loss_parts=loss_parts,frozen_modules=list(frozen),
                    seconds_per_microbatch=float(np.median(timings[1:])),
                    peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/2**30,
                    elapsed_seconds=time.perf_counter()-started,ema_updates=ema.updates,
                    config_sha256=file_sha256(f'experiments/configs/{config["name"]}.json'))
        results.append(result)
        atomic_json('artifacts/refine_integration_checks.json',dict(manifest_sha256=manifest,
                    split_pixel_overlap=0,paired_config_differences=sorted(differences),runs=results))
        print(result,flush=True)
        del model,optimizer,ema,head,before,images,labels,targets,logits,loss,ce_loss,frozen,frozen_before
        gc.collect()
        torch.cuda.empty_cache()
    print('All refinement GPU checks passed. Temporary updates were discarded.',flush=True)


if __name__=='__main__':
    main()
""",

'fer/verify_sft.py': """\
\"\"\"Exercise every planned fine-tuning configuration on real training batches.\"\"\"
import gc
import hashlib
import json
from pathlib import Path
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from .data import FERDataset, ImagePipeline
from .io import atomic_json, read_json
from .models import make_model
from .train import build_optimizer, check_gradient_finite, evaluate, forward_loss, mixup_batch, seed_everything


def main():
    plan=read_json('experiments/sft_plan.json')
    data=FERDataset(plan['data'],'train')
    val=FERDataset(plan['data'],'val')
    assert set(data.indices).isdisjoint(val.indices)
    assert {data.rows[i]['pixel_sha256'] for i in data.indices}.isdisjoint({val.rows[i]['pixel_sha256'] for i in val.indices})
    manifest=hashlib.sha256((Path(plan['data'])/'manifest.jsonl').read_bytes()).hexdigest()
    assert manifest==data.split['manifest_sha256']
    results=[]
    for path in plan['configs']:
        config=read_json(path)
        seed_everything(config['seed'])
        model,info=make_model(config,load_pretrained=not bool(config.get('finetune_from')))
        if config.get('finetune_from'):
            source=torch.load(config['finetune_from'],map_location='cpu',weights_only=True)
            assert source['config']['manifest_sha256']==manifest
            assert source['classes']==data.split['classes']
            model.load_state_dict(source['model'],strict=True)
            del source
        for stage in range(config.get('freeze_stages',0)):
            model.stages[stage].requires_grad_(False)
            model.downsample_layers[stage].requires_grad_(False)
        model=model.to(device='cuda',memory_format=torch.channels_last).train()
        optimizer=build_optimizer(model,config)
        pipeline=ImagePipeline('cuda',augmentation=config.get('augmentation'))
        loader=DataLoader(Subset(data,range(config['batch_size']*4)),batch_size=config['batch_size'])
        before=model.head.weight.detach().clone()
        frozen_before=model.downsample_layers[0][0].weight.detach().clone()
        torch.cuda.reset_peak_memory_stats()
        timings=[]
        for raw,labels,_ in loader:
            started=time.perf_counter()
            labels=labels.to('cuda')
            images=pipeline(raw.to('cuda'),training=True)
            images,targets=mixup_batch(images,labels,config.get('mixup_alpha',0.),1.)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.bfloat16):
                loss,logits=forward_loss(model,images,targets,config,0)
            assert torch.isfinite(loss) and logits.shape==(len(labels),6)
            loss.backward()
            check_gradient_finite(model)
            optimizer.step()
            torch.cuda.synchronize()
            timings.append(time.perf_counter()-started)
        assert not torch.equal(before,model.head.weight)
        if config.get('freeze_stages',0):
            torch.testing.assert_close(frozen_before,model.downsample_layers[0][0].weight,rtol=0,atol=0)
            assert model.downsample_layers[0][0].weight.grad is None
        val_loader=DataLoader(Subset(val,range(16)),batch_size=16)
        y,probs,indices=evaluate(model,val_loader,pipeline,config['model'],'cuda',precision='float32')
        assert np.isfinite(probs).all()
        np.testing.assert_allclose(probs.sum(1),1.,atol=1e-6)
        np.testing.assert_array_equal(indices,val.indices[:16])
        result=dict(name=config['name'],status='passed',pretrained=info,
                    parameters=sum(p.numel() for p in model.parameters()),
                    trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                    seconds_per_microbatch=float(np.median(timings[1:])),
                    peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/2**30)
        results.append(result)
        print(json.dumps(result),flush=True)
        atomic_json('artifacts/sft_integration_checks.json',dict(manifest_sha256=manifest,split_overlap=0,runs=results))
        del model,optimizer,images,targets,loss,logits,before,frozen_before
        gc.collect()
        torch.cuda.empty_cache()
    print('All fine-tuning GPU checks passed.',flush=True)


if __name__=='__main__':
    main()
""",

'fer/verify_transfer.py': """\
\"\"\"Preflight on real train batches, without retaining these temporary updates.\"\"\"
import argparse
import gc
import hashlib
from pathlib import Path
import time

import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from .data import FERDataset, ImagePipeline
from .io import atomic_json, read_json
from .losses import eac_consistency
from .models import make_model
from .train import build_optimizer, check_gradient_finite, evaluate, forward_loss, seed_everything


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--only',choices=['ce','eac'])
    args = parser.parse_args()
    plan = read_json('experiments/transfer_plan.json')
    training, validation = (FERDataset(plan['data'], split) for split in ('train','val'))
    assert set(training.indices).isdisjoint(validation.indices)
    assert {training.rows[i]['pixel_sha256'] for i in training.indices}.isdisjoint(
        {validation.rows[i]['pixel_sha256'] for i in validation.indices})
    manifest = hashlib.sha256((Path(plan['data'])/'manifest.jsonl').read_bytes()).hexdigest()
    assert manifest == training.split['manifest_sha256']
    configs = [read_json(path) for path in plan['configs']]
    ce, eac = configs[:2]
    different = {key for key in ce.keys() | eac.keys() if ce.get(key) != eac.get(key)}
    assert different == {'name','output','loss','eac_weight'}, different
    results = []
    for config in configs:
        if args.only and config['loss'] != args.only:
            continue
        seed_everything(config['seed'])
        model, _ = make_model(config)
        model = model.to(device='cuda',memory_format=torch.channels_last).train()
        optimizer = build_optimizer(model,config)
        pipeline = ImagePipeline('cuda',size=config['image_size'],augmentation=config['augmentation'])
        loader = DataLoader(Subset(training,range(config['batch_size']*5)),batch_size=config['batch_size'])
        before = model.backbone.classifier.weight.detach().clone()
        timings, losses = [], []
        torch.cuda.reset_peak_memory_stats()
        first_loss_parts = None
        for raw,labels,_ in loader:
            started = time.perf_counter()
            images, labels = pipeline(raw.to('cuda'),training=True), labels.to('cuda')
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.bfloat16):
                loss, logits = forward_loss(model,images,labels,config,0)
            if first_loss_parts is None:
                ce_value = F.cross_entropy(logits.float(),labels,label_smoothing=config['label_smoothing'])
                first_loss_parts = dict(ce=float(ce_value.detach()),weighted_consistency=float((loss-ce_value).detach()))
            assert torch.isfinite(loss) and logits.shape == (len(labels),6)
            loss.backward()
            check_gradient_finite(model)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
            optimizer.step()
            torch.cuda.synchronize()
            timings.append(time.perf_counter()-started)
            losses.append(float(loss.detach()))
        assert not torch.equal(before,model.backbone.classifier.weight)
        labels,probs,indices = evaluate(model,DataLoader(Subset(validation,range(32)),batch_size=32),
                                        pipeline,config['model'],'cuda',precision='float32')
        assert np.isfinite(probs).all()
        np.testing.assert_allclose(probs.sum(1),1.,atol=1e-6)
        np.testing.assert_array_equal(indices,validation.indices[:32])
        result = dict(name=config['name'],status='passed',initial_loss_parts=first_loss_parts,
                      losses=losses,seconds_per_microbatch=float(np.median(timings[1:])),
                      peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/2**30,
                      parameters=sum(p.numel() for p in model.parameters()))
        results.append(result)
        print(result,flush=True)
        output = f'artifacts/transfer_integration_{args.only}.json' if args.only else 'artifacts/transfer_integration_checks.json'
        atomic_json(output,dict(manifest_sha256=manifest,
                    split_pixel_overlap=0,paired_config_differences=sorted(different),runs=results))
        del model, optimizer, images, labels, logits, loss, before, ce_value
        gc.collect()
        torch.cuda.empty_cache()
    print('All transfer GPU checks passed.',flush=True)


if __name__ == '__main__':
    main()
""",
})


In [ ]:
write_sources({
'fer/train.py': """\
\"\"\"Train the controlled six-class FER experiments and retain validation evidence.\"\"\"
import argparse
import copy
from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import random
import time

import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from .data import FERDataset, ImagePipeline
from .io import atomic_json, atomic_output, update_status
from .losses import eac_consistency, nla_loss
from .models import make_model
from .refinement import ReplacementReference, freeze_modules


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(8)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def atomic_checkpoint(path, value):
    with atomic_output(path) as temp:
        torch.save(value, temp)


class EMA:
    def __init__(self, model):
        self.model = copy.deepcopy(model).eval()
        self.model.requires_grad_(False)
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        decay = min(.999, (1+self.updates)/(10+self.updates))
        original = model.state_dict()
        averaged, current = [], []
        for key, target in self.model.state_dict().items():
            source = original[key].detach()
            if target.is_floating_point():
                averaged.append(target)
                current.append(source)
            else:
                target.copy_(source)
        torch._foreach_lerp_(averaged, current, 1-decay)


def classification_metrics(labels, probabilities, classes):
    prediction = probabilities.argmax(1)
    return dict(accuracy=float(accuracy_score(labels,prediction)),
                macro_f1=float(f1_score(labels,prediction,labels=range(6),average='macro',zero_division=0)),
                confusion_matrix=confusion_matrix(labels,prediction,labels=range(6)).tolist(),
                per_class=classification_report(labels,prediction,labels=range(6),target_names=classes,
                                                output_dict=True,zero_division=0))


@contextmanager
def evaluation_precision(precision):
    if precision not in ('bf16','float32'):
        raise ValueError(f'Unknown evaluation precision: {precision}')
    if precision=='bf16':
        yield
        return
    previous = torch.backends.cuda.matmul.allow_tf32
    try:
        torch.backends.cuda.matmul.allow_tf32 = False
        with torch.backends.cudnn.flags(benchmark=False,deterministic=True,allow_tf32=False):
            yield
    finally:
        torch.backends.cuda.matmul.allow_tf32 = previous


@torch.inference_mode()
def evaluate(model, loader, pipeline, model_name, device, tta=False, precision='bf16'):
    model.eval()
    probabilities, targets, indices = [], [], []
    with evaluation_precision(precision):
        for raw, labels, row_indices in loader:
            images = pipeline(raw.to(device, non_blocking=True), training=False)
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=precision=='bf16'):
                logits = model(images,None,None,None) if model_name=='qcs' else model(images)
                probs = logits.float().softmax(1)
                if tta:
                    flipped = images.flip(-1)
                    logits2 = model(flipped,None,None,None) if model_name=='qcs' else model(flipped)
                    probs = .5*(probs+logits2.float().softmax(1))
            probabilities.append(probs.cpu().numpy())
            targets.append(labels.numpy())
            indices.append(row_indices.numpy())
    return np.concatenate(targets), np.concatenate(probabilities), np.concatenate(indices)


def build_optimizer(model, config):
    if config['model'] == 'emotieff':
        groups = {}
        for name, parameter in model.named_parameters():
            if not parameter.requires_grad:
                continue
            scale = config.get('head_lr_multiplier',1.) if name.startswith('backbone.classifier.') else 1.
            decay = 0. if parameter.ndim <= 1 or name.endswith('.bias') else config['weight_decay']
            group = groups.setdefault((scale,decay),dict(params=[],lr=config['lr']*scale,
                                                        lr_scale=scale,weight_decay=decay))
            group['params'].append(parameter)
        return torch.optim.AdamW(list(groups.values()),lr=config['lr'])
    if config['model'] == 'qcs':
        from third_party.qcs.sam import SAM
        return SAM(model.parameters(), torch.optim.Adam, lr=config['lr'], rho=.05,
                   weight_decay=config.get('weight_decay',0))
    if 'layer_decay' not in config:
        return torch.optim.AdamW(model.parameters(),lr=config['lr'],weight_decay=config['weight_decay'])
    groups = {}
    for name,parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        layer = int(name.split('.')[1]) if name.startswith(('stages.','downsample_layers.')) else 4
        scale = config['layer_decay']**(4-layer)
        decay = 0. if parameter.ndim<=1 or name.endswith('.bias') else config['weight_decay']
        group = groups.setdefault((layer,decay),dict(params=[],lr=config['lr']*scale,
                                                    lr_scale=scale,weight_decay=decay))
        group['params'].append(parameter)
    return torch.optim.AdamW(list(groups.values()),lr=config['lr'])


def mixup_batch(images, labels, alpha, probability):
    if alpha<=0 or len(labels)<2 or np.random.random()>=probability:
        return images, labels
    weight = float(np.random.beta(alpha,alpha))
    permutation = torch.randperm(len(labels),device=labels.device)
    targets = F.one_hot(labels,num_classes=6).float()
    return (images*weight+images[permutation]*(1-weight),
            targets*weight+targets[permutation]*(1-weight))


def selection_key(metrics, selection_metric):
    if selection_metric=='validation_accuracy':
        return metrics['accuracy'],metrics['macro_f1']
    if selection_metric=='validation_macro_f1':
        return metrics['macro_f1'],metrics['accuracy']
    raise ValueError(f'Unknown selection metric: {selection_metric}')


def save_best(run, model, config, epoch, metrics, labels, probabilities, indices, classes,
              selection_metrics=None):
    selection_metrics = selection_metrics or metrics
    atomic_checkpoint(run/'best.pt',dict(model=model.state_dict(),config=config,
                      epoch=epoch,metrics=metrics,classes=classes,selection_metrics=selection_metrics))
    with atomic_output(run/'val_predictions.npz') as temp:
        with temp.open('wb') as file:
            np.savez_compressed(file,labels=labels,probabilities=probabilities,indices=indices)
    atomic_json(run/'best_metrics.json',dict(epoch=epoch,**metrics,selection_metrics=selection_metrics))


def forward_loss(model, images, labels, config, epoch, negative_labels=None):
    if config.get('loss') == 'eac':
        logits, attention = model(images,return_attention=True)
        # The unused flipped classifier must not draw a second dropout mask:
        # keep the augmentation/dropout random stream aligned with the CE run.
        flipped_attention = model(images.flip(-1),attention_only=True)
        loss = F.cross_entropy(logits.float(),labels,label_smoothing=config['label_smoothing'])
        return loss + config['eac_weight'] * eac_consistency(attention,flipped_attention), logits
    if config['model'] == 'qcs':
        outputs = model(images[0],images[1],images[2],images[3])
        all_labels = [labels,labels,negative_labels,negative_labels]*2
        loss = sum(F.cross_entropy(logits.float(),target) for logits,target in zip(outputs,all_labels))/8
        return loss, outputs[0]
    logits, regularizer = model(images,return_aux=True)
    if config['loss'] == 'nla':
        flipped = model(images.flip(-1))
        loss = nla_loss(logits,flipped,labels,epoch,config['epochs'],config['effective_batch_size'])
    else:
        loss = F.cross_entropy(logits.float(),labels,label_smoothing=config['label_smoothing'])
    return loss + regularizer.float(), logits


def check_gradient_finite(model):
    for name, parameter in model.named_parameters():
        if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
            raise FloatingPointError(f'Nonfinite gradient in {name}')


def train(config_path, resume=False):
    config_path = Path(config_path)
    config = json.loads(config_path.read_text(encoding='utf-8'))
    run = Path(config['output'])
    if (run/'metrics.jsonl').exists() and not resume:
        raise FileExistsError(f'Existing experiment: {run}; use --resume or a new run name.')
    run.mkdir(parents=True,exist_ok=True)
    seed_everything(config['seed'])
    if not torch.cuda.is_available():
        raise RuntimeError('A CUDA GPU is required for this experiment budget.')
    device = torch.device('cuda:0')
    training = FERDataset(config['data'],'train',quadruplets=config['model']=='qcs')
    validation = FERDataset(config['data'],'val')
    manifest_bytes = (Path(config['data'])/'manifest.jsonl').read_bytes()
    assert hashlib.sha256(manifest_bytes).hexdigest() == training.split['manifest_sha256']
    config['manifest_sha256'] = training.split['manifest_sha256']
    atomic_json(run/'config.json',config)
    loader_generator = torch.Generator().manual_seed(config['seed'])
    train_loader = DataLoader(training,batch_size=config['batch_size'],shuffle=True,
                              num_workers=0,pin_memory=True,drop_last=False,generator=loader_generator)
    val_loader = DataLoader(validation,batch_size=config.get('eval_batch_size',64),shuffle=False,
                            num_workers=0,pin_memory=True)
    pipeline = ImagePipeline(device,size=config.get('image_size',224),augmentation=config.get('augmentation'))
    source_path = config.get('finetune_from')
    model, pretrained_info = make_model(config,load_pretrained=not (source_path or resume))
    if config['model']=='emotieff' and pretrained_info['classes']!=training.split['classes']:
        raise RuntimeError('Training class order differs from the mapped EmotiEff classifier.')
    if source_path and not resume:
        source = torch.load(source_path,map_location='cpu',weights_only=True)
        if source['config']['manifest_sha256']!=config['manifest_sha256'] or source['classes']!=training.split['classes']:
            raise RuntimeError('Fine-tuning checkpoint has a different data manifest or label mapping.')
        model.load_state_dict(source['model'],strict=True)
        pretrained_info.update(finetune_from=source_path,source_epoch=source['epoch'],
                               checkpoint_sha256=hashlib.sha256(Path(source_path).read_bytes()).hexdigest())
        del source
    frozen_modules = freeze_modules(model,config)
    reference = None
    if config.get('selection_reference'):
        if config.get('eval_precision') != 'float32' or config.get('eval_batch_size') != 32:
            raise ValueError('Replacement selection requires FP32 evaluation with batch size 32.')
        spec = config['selection_reference']
        if not source_path or Path(source_path).resolve() != Path(spec['replaces_checkpoint']).resolve():
            raise ValueError('Replacement parent must match the fine-tuning source.')
        reference = ReplacementReference(spec,validation.indices,validation.labels,
                                         validation.split['classes'],config['manifest_sha256'])
    def selection_metrics_for(labels,probs,indices,metrics):
        if reference is None:
            return metrics
        return classification_metrics(labels,reference.combine(labels,probs,indices),training.split['classes'])
    model = model.to(device=device,memory_format=torch.channels_last)
    optimizer = build_optimizer(model,config)
    ema = EMA(model) if config.get('ema',False) else None
    parameters = sum(p.numel() for p in model.parameters())
    atomic_json(run/'initialization.json',dict(pretrained=pretrained_info,parameters=parameters,
                gpu=torch.cuda.get_device_name(0),torch=torch.__version__,numpy=np.__version__,
                config_file=str(config_path),started_at=datetime.now(timezone.utc).isoformat()))
    start_epoch, best_f1, best_acc, best_epoch = 0, -1., -1., -1
    best_selection = dict(accuracy=-1.,macro_f1=-1.)
    selection_scope = 'replacement_ensemble' if reference is not None else 'single_model'
    total_steps = 0
    bad_epochs = 0
    selection_metric = config.get('selection_metric','validation_macro_f1')
    eval_precision = config.get('eval_precision','bf16')
    selection_key(dict(accuracy=0.,macro_f1=0.),selection_metric)
    if resume:
        checkpoint = torch.load(run/'last.pt',map_location='cpu',weights_only=False)
        if checkpoint['config']['manifest_sha256'] != config['manifest_sha256']:
            raise RuntimeError('Cannot resume with a different manifest.')
        if checkpoint['config'] != config:
            raise RuntimeError('Resume configuration changed; create a new fine-tuning run instead.')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        if ema is not None:
            ema.model.load_state_dict(checkpoint['ema'])
            ema.updates = checkpoint['ema_updates']
        start_epoch=checkpoint['epoch']+1
        best_f1,best_acc,best_epoch=checkpoint['best_f1'],checkpoint['best_acc'],checkpoint['best_epoch']
        best_selection=checkpoint.get('best_selection',dict(accuracy=best_acc,macro_f1=best_f1))
        total_steps=checkpoint['total_steps']
        bad_epochs=checkpoint.get('bad_epochs',0)
        random.setstate(checkpoint['python_rng'])
        np.random.set_state(checkpoint['numpy_rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        loader_generator.set_state(checkpoint['loader_rng'])
        del checkpoint
    elif source_path or config.get('evaluate_initial',False):
        update_status(run/'status.json',dict(status='initial_evaluation',run=config['name'],
                      eval_precision=eval_precision))
        labels,probs,indices=evaluate(ema.model if ema is not None else model,val_loader,pipeline,
                                      config['model'],device,precision=eval_precision)
        metrics=classification_metrics(labels,probs,training.split['classes'])
        best_selection=selection_metrics_for(labels,probs,indices,metrics)
        if reference is not None and abs(best_selection['accuracy']-config['selection_reference']['baseline_accuracy'])>1e-12:
            raise RuntimeError('Initial replacement ensemble does not reproduce the reference accuracy.')
        best_f1,best_acc,best_epoch=metrics['macro_f1'],metrics['accuracy'],0
        save_best(run,ema.model if ema is not None else model,config,0,metrics,
                  labels,probs,indices,training.split['classes'],best_selection)
        atomic_json(run/'initial_metrics.json',dict(**metrics,selection_metrics=best_selection))
        print(json.dumps(dict(event='initial_evaluation',run=config['name'],accuracy=best_acc,
                              macro_f1=best_f1,selection_accuracy=best_selection['accuracy'],
                              selection_scope=selection_scope)),flush=True)
    started = time.time()
    completed_epochs = start_epoch
    stopped_early = False
    print(json.dumps(dict(event='start',run=config['name'],train=len(training),val=len(validation),
                          parameters=parameters,start_epoch=start_epoch)),flush=True)
    try:
        for epoch in range(start_epoch,config['epochs']):
            model.train()
            for module in frozen_modules.values():
                module.eval()
            warmup = min(1.,(epoch+1)/config.get('warmup_epochs',1))
            if config['model']=='qcs':
                lr = config['lr'] * .98**epoch
            else:
                fraction=max(0.,(epoch-config.get('warmup_epochs',1)+1)/max(1,config['epochs']-config.get('warmup_epochs',1)))
                lr=config['lr']*warmup*(.05+.95*.5*(1+math.cos(math.pi*fraction)))
            for group in optimizer.param_groups:
                group['lr']=lr*group.get('lr_scale',1.)
            epoch_start=time.time()
            loss_sum=correct=count=0
            optimizer.zero_grad(set_to_none=True)
            accumulation=config.get('accumulation_steps',1)
            torch.cuda.reset_peak_memory_stats()
            for batch_index,batch in enumerate(train_loader):
                if config['model']=='qcs':
                    raw,labels,negative_labels=batch
                    labels=labels.to(device,non_blocking=True)
                    negative_labels=negative_labels.to(device,non_blocking=True)
                    raw=raw.to(device,non_blocking=True)
                    images=[pipeline(raw[:,i],training=True) for i in range(4)]
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        loss,logits=forward_loss(model,images,labels,config,epoch,negative_labels)
                    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite QCS first-step loss')
                    loss.backward()
                    if batch_index==0: check_gradient_finite(model)
                    optimizer.first_step(zero_grad=True)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        sam_loss,_=forward_loss(model,images,labels,config,epoch,negative_labels)
                    if not torch.isfinite(sam_loss): raise FloatingPointError('Nonfinite QCS SAM loss')
                    sam_loss.backward()
                    if batch_index==0: check_gradient_finite(model)
                    optimizer.second_step(zero_grad=True)
                    total_steps+=1
                else:
                    raw,labels,_=batch
                    labels=labels.to(device,non_blocking=True)
                    images=pipeline(raw.to(device,non_blocking=True),training=True)
                    images,loss_labels=mixup_batch(images,labels,config.get('mixup_alpha',0.),
                                                   config.get('mixup_probability',0.))
                    group_start=(batch_index//accumulation)*accumulation
                    group_samples=min(config['batch_size']*accumulation,len(training)-group_start*config['batch_size'])
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        loss,logits=forward_loss(model,images,loss_labels,config,epoch)
                    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite training loss')
                    (loss*len(labels)/group_samples).backward()
                    if (batch_index+1)%accumulation==0 or batch_index+1==len(train_loader):
                        if batch_index<accumulation: check_gradient_finite(model)
                        torch.nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
                        optimizer.step()
                        optimizer.zero_grad(set_to_none=True)
                        total_steps+=1
                        if ema is not None: ema.update(model)
                loss_sum+=loss.item()*len(labels)
                prediction=logits.detach().argmax(1)
                if config['model']!='qcs' and loss_labels.ndim==2:
                    correct+=loss_labels.gather(1,prediction[:,None]).sum().item()
                else:
                    correct+=prediction.eq(labels).sum().item()
                count+=len(labels)
                if batch_index==0 or (batch_index+1)%100==0:
                    status=dict(status='training',run=config['name'],epoch=epoch+1,epochs=config['epochs'],
                                batch=batch_index+1,batches=len(train_loader),train_loss=loss_sum/count,
                                train_accuracy=correct/count,epoch_seconds=time.time()-epoch_start,
                                best_val_macro_f1=best_f1,best_val_accuracy=best_acc,
                                best_selection_accuracy=best_selection['accuracy'],selection_scope=selection_scope,
                                updated_at=datetime.now(timezone.utc).isoformat())
                    update_status(run/'status.json',status)
                    print(json.dumps(status),flush=True)
            eval_model=ema.model if ema is not None else model
            labels,probs,indices=evaluate(eval_model,val_loader,pipeline,config['model'],device,
                                          precision=eval_precision)
            metrics=classification_metrics(labels,probs,training.split['classes'])
            scoring=selection_metrics_for(labels,probs,indices,metrics)
            record=dict(epoch=epoch+1,lr=lr,train_loss=loss_sum/count,train_accuracy=correct/count,
                        val_accuracy=metrics['accuracy'],val_macro_f1=metrics['macro_f1'],
                        selection_accuracy=scoring['accuracy'],selection_macro_f1=scoring['macro_f1'],
                        selection_scope=selection_scope,
                        epoch_seconds=time.time()-epoch_start,total_seconds=time.time()-started,
                        peak_gpu_memory_gib=torch.cuda.max_memory_allocated()/2**30,
                        train_accuracy_kind='mixup_weighted' if config.get('mixup_alpha',0.) else 'hard_labels')
            with (run/'metrics.jsonl').open('a',encoding='utf-8') as file:
                file.write(json.dumps(record)+'\\n')
            is_best=selection_key(scoring,selection_metric)>selection_key(best_selection,selection_metric)
            bad_epochs=0 if is_best else bad_epochs+1
            if is_best:
                best_f1,best_acc,best_epoch=metrics['macro_f1'],metrics['accuracy'],epoch+1
                best_selection=scoring
                save_best(run,eval_model,config,epoch+1,metrics,labels,probs,indices,training.split['classes'],scoring)
            checkpoint=dict(model=model.state_dict(),optimizer=optimizer.state_dict(),config=config,
                            epoch=epoch,best_f1=best_f1,best_acc=best_acc,best_epoch=best_epoch,total_steps=total_steps,
                            bad_epochs=bad_epochs,best_selection=best_selection,
                            python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all(),loader_rng=loader_generator.get_state())
            if ema is not None: checkpoint.update(ema=ema.model.state_dict(),ema_updates=ema.updates)
            atomic_checkpoint(run/'last.pt',checkpoint)
            del checkpoint
            update_status(run/'status.json',dict(status='epoch_complete',run=config['name'],**record,
                        best_epoch=best_epoch,best_val_accuracy=best_acc,best_val_macro_f1=best_f1,
                        best_selection_accuracy=best_selection['accuracy']))
            print(json.dumps(dict(event='epoch_complete',run=config['name'],**record,is_best=is_best)),flush=True)
            completed_epochs=epoch+1
            patience=config.get('early_stopping_patience',0)
            if patience and bad_epochs>=patience and completed_epochs>=config.get('min_epochs',0):
                stopped_early=True
                print(json.dumps(dict(event='early_stopping',epoch=completed_epochs,best_epoch=best_epoch)),flush=True)
                break
        update_status(run/'status.json',dict(status='complete',run=config['name'],epochs=config['epochs'],
                    best_epoch=best_epoch,best_val_accuracy=best_acc,best_val_macro_f1=best_f1,
                    best_selection_accuracy=best_selection['accuracy'],selection_scope=selection_scope,
                    epochs_completed=completed_epochs,stopped_early=stopped_early,selection_metric=selection_metric,
                    total_seconds=time.time()-started,finished_at=datetime.now(timezone.utc).isoformat()),required=True)
        print(json.dumps(dict(event='complete',run=config['name'],best_epoch=best_epoch,
                              best_val_accuracy=best_acc,best_val_macro_f1=best_f1)),flush=True)
    except BaseException as error:
        update_status(run/'status.json',dict(status='failed',run=config['name'],error=repr(error),
                    updated_at=datetime.now(timezone.utc).isoformat()))
        raise


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--config',required=True)
    parser.add_argument('--resume',action='store_true')
    args=parser.parse_args()
    train(args.config,args.resume)


if __name__=='__main__':
    main()
""",

'fer/finalize_sft.py': """\
\"\"\"Re-evaluate supervised fine-tuning candidates, select a recipe and export.\"\"\"
import argparse
import csv
from datetime import datetime, timedelta, timezone
import gc
import hashlib
import json
from pathlib import Path
import traceback

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import FERDataset, ImagePipeline
from .io import atomic_json, atomic_output, read_json, update_status
from .models import make_model
from .selection import recipe_candidates
from .train import classification_metrics, evaluate, seed_everything


STATUS=Path('artifacts/sft_finalization_status.json')


def file_hash(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as file:
        for block in iter(lambda:file.read(4*1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def save_npz(path, **arrays):
    with atomic_output(path) as temp:
        with temp.open('wb') as file:
            np.savez_compressed(file,**arrays)


def load_model(profile, manifest, classes):
    checkpoint=torch.load(profile['checkpoint'],map_location='cpu',weights_only=True)
    config=checkpoint['config']
    if config['manifest_sha256']!=manifest or checkpoint['classes']!=classes:
        raise RuntimeError(f'Checkpoint data/label mismatch: {profile["checkpoint"]}')
    model,_=make_model(config,load_pretrained=False)
    model.load_state_dict(checkpoint['model'],strict=True)
    del checkpoint
    return model.to(device='cuda',memory_format=torch.channels_last).eval(),config


def validation_predictions(profile, dataset, source_hashes, cache_directory='artifacts/sft_validation'):
    cache=Path(cache_directory)
    cache.mkdir(parents=True,exist_ok=True)
    key=profile['key']
    metadata={**profile,'manifest_sha256':dataset.split['manifest_sha256'],
              'eval_precision':'float32','eval_batch_size':32,'source_sha256':source_hashes}
    npz_path=cache/(key+'.npz')
    meta_path=cache/(key+'.json')
    if npz_path.exists() and meta_path.exists() and read_json(meta_path)==metadata:
        with np.load(npz_path) as saved:
            labels=saved['labels'].copy()
            probs=saved['probabilities'].copy()
            indices=saved['indices'].copy()
    else:
        model,config=load_model(profile,dataset.split['manifest_sha256'],dataset.split['classes'])
        loader=DataLoader(dataset,batch_size=32,num_workers=0,pin_memory=True)
        pipeline=ImagePipeline('cuda',size=config.get('image_size',224))
        labels,probs,indices=evaluate(model,loader,pipeline,config['model'],'cuda',
                                      tta=profile['flip_tta'],precision='float32')
        save_npz(npz_path,labels=labels,probabilities=probs,indices=indices)
        atomic_json(meta_path,metadata)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    np.testing.assert_array_equal(indices,dataset.indices)
    np.testing.assert_array_equal(labels,dataset.labels)
    if probs.shape!=(len(dataset),6) or not np.isfinite(probs).all():
        raise RuntimeError('Invalid validation probabilities.')
    np.testing.assert_allclose(probs.sum(1),1.,atol=1e-6)
    return probs


def main():
    global STATUS
    parser=argparse.ArgumentParser()
    parser.add_argument('--plan',default='experiments/sft_plan.json')
    args=parser.parse_args()
    plan=read_json(args.plan)
    prefix=plan.get('result_prefix','sft')
    STATUS=Path(f'artifacts/{prefix}_finalization_status.json')
    cache_directory=f'artifacts/{prefix}_validation'
    seed_everything(plan['seed'])
    validation=FERDataset(plan['data'],'val')
    classes=validation.split['classes']
    manifest=file_hash(Path(plan['data'])/'manifest.jsonl')
    if manifest!=validation.split['manifest_sha256']:
        raise RuntimeError('Data manifest was modified.')
    source_hashes={name:file_hash(name) for name in ['fer/train.py','fer/models.py','fer/data.py',
                  'fer/emotieff.py','fer/losses.py','fer/finalize_sft.py','fer/selection.py','fer/refinement.py']}
    profiles={}
    probabilities={}
    records=[]
    for path in plan['reference_configs']+plan.get('comparison_configs',[])+plan['configs']:
        config=read_json(path)
        run=Path(config['output'])
        status=read_json(run/'status.json')
        if status.get('status')!='complete':
            raise RuntimeError(f'Training has not completed: {run}')
        history=[json.loads(line) for line in (run/'metrics.jsonl').read_text(encoding='utf-8').splitlines()]
        expected=status.get('epochs_completed',config['epochs'])
        if [row['epoch'] for row in history]!=list(range(1,expected+1)):
            raise RuntimeError(f'Incomplete or duplicated epoch history: {run}')
        checkpoint=run/'best.pt'
        profile=dict(key=config['name'],run=config['name'],config_path=path,model=config['model'],
                     checkpoint=str(checkpoint),checkpoint_sha256=file_hash(checkpoint),flip_tta=False)
        profiles[profile['key']]=profile
        update_status(STATUS,dict(status='validating',current=config['name']))
        probs=validation_predictions(profile,validation,source_hashes,cache_directory)
        probabilities[profile['key']]=probs
        metrics=classification_metrics(validation.labels,probs,classes)
        record=dict(name=config['name'],phase=prefix if path in plan['configs'] else 'reference',
                    accuracy=metrics['accuracy'],macro_f1=metrics['macro_f1'],
                    best_epoch=read_json(run/'best_metrics.json')['epoch'],epochs_completed=expected,
                    early_stopped=status.get('stopped_early',False),
                    training_seconds=sum(row['epoch_seconds'] for row in history),
                    per_class=metrics['per_class'])
        records.append(record)
        atomic_json(f'artifacts/{prefix}_revalidation.json',records)
        print(json.dumps(record),flush=True)
    best_plain=max(records,key=lambda r:(r['accuracy'],r['macro_f1']))
    flip={**profiles[best_plain['name']],'key':best_plain['name']+'_flip','flip_tta':True}
    profiles[flip['key']]=flip
    update_status(STATUS,dict(status='checking_flip_tta',current=best_plain['name']))
    probabilities[flip['key']]=validation_predictions(flip,validation,source_hashes,cache_directory)
    reference_keys=[read_json(path)['name'] for path in plan['reference_configs']]
    new_keys=[read_json(path)['name'] for path in plan['configs']]
    candidates=recipe_candidates(probabilities,validation.labels,profiles,plan['ensemble_weights'],reference_keys,
                                 new_keys,plan.get('reference_blend_weights',[]),plan.get('replacement_parents'))
    selected=candidates[0]
    selected_probs=sum(probabilities[c['profile']]*c['weight'] for c in selected['components'])
    selected_metrics=classification_metrics(validation.labels,selected_probs,classes)
    reference_probs=sum(probabilities[key]/len(reference_keys) for key in reference_keys)
    reference_metrics=classification_metrics(validation.labels,reference_probs,classes)
    previous=None
    if plan.get('previous_result'):
        previous=read_json(plan['previous_result'])['selected_validation_metrics']
        if abs(reference_metrics['accuracy']-previous['accuracy'])>1/len(validation):
            raise RuntimeError('Previous best ensemble could not be reproduced to within one validation image.')
    best_single=next(c for c in candidates if len(c['components'])==1)
    best_sft=max((row for row in records if row['phase']==prefix),key=lambda r:(r['accuracy'],r['macro_f1']))
    baseline=next(row for row in records if row['name']=='convnext_tiny_s42')
    output=Path(plan.get('prediction_directory','predictions/sft'))
    output.mkdir(parents=True,exist_ok=True)
    save_npz(output/'selected_validation_predictions.npz',labels=validation.labels,
             probabilities=selected_probs,indices=np.array(validation.indices))
    testing=FERDataset(plan['data'],'test')
    test_probs=np.zeros((len(testing),6),dtype=np.float32)
    components=[]
    for component in selected['components']:
        profile=profiles[component['profile']]
        update_status(STATUS,dict(status='predicting_test',current=profile['run'],recipe=selected['name']))
        model,config=load_model(profile,manifest,classes)
        loader=DataLoader(testing,batch_size=32,num_workers=0,pin_memory=True)
        _,probs,indices=evaluate(model,loader,ImagePipeline('cuda',size=config.get('image_size',224)),
                                 config['model'],'cuda',tta=profile['flip_tta'],precision='float32')
        np.testing.assert_array_equal(indices,testing.indices)
        test_probs+=probs*component['weight']
        components.append({**profile,'weight':component['weight'],'image_size':config.get('image_size',224)})
        del model
        gc.collect()
        torch.cuda.empty_cache()
    if test_probs.shape!=(7065,6) or not np.isfinite(test_probs).all():
        raise RuntimeError('Invalid test probabilities.')
    np.testing.assert_allclose(test_probs.sum(1),1.,atol=1e-5)
    with atomic_output(output/'best_predictions.csv') as temp:
        with temp.open('w',encoding='utf-8',newline='') as file:
            writer=csv.writer(file)
            writer.writerow(['image','predicted_label','class_id','confidence'])
            for index,probs in zip(testing.indices,test_probs):
                label=int(probs.argmax())
                writer.writerow([Path(testing.rows[index]['path']).name,classes[label],label,f'{float(probs[label]):.8f}'])
    with (output/'best_predictions.csv').open(encoding='utf-8',newline='') as file:
        exported=list(csv.DictReader(file))
    assert len(exported)==len({row['image'] for row in exported})==7065
    save_npz(output/'test_probabilities.npz',probabilities=test_probs,indices=np.array(testing.indices),classes=np.array(classes))
    recipe=dict(name=selected['name'],components=components,classes=classes,eval_precision='float32',
                manifest_sha256=manifest,selection_metric='validation_accuracy',source_sha256=source_hashes)
    atomic_json(output/'inference_recipe.json',recipe)
    summary=dict(completed_at=datetime.now(timezone.utc).isoformat(),plan=args.plan,
                 selected_model=selected['name'],selected_recipe=recipe,selected_validation_metrics=selected_metrics,
                 best_single=best_single,best_sft_single=best_sft,baseline=baseline,runs=records,candidates=candidates,
                 accuracy_gain_percentage_points=100*(selected_metrics['accuracy']-baseline['accuracy']),
                 validation_images=len(validation),test_images=len(testing),test_labels_available=False,
                 predictions=(output/'best_predictions.csv').as_posix(),
                 limitations=['One existing validation split is reused for tuning, early stopping and recipe selection.',
                              'Validation improvements are not independent labeled test results.'])
    summary['reference_ensemble_metrics']=reference_metrics
    summary['gain_over_reference_ensemble_pp']=100*(selected_metrics['accuracy']-reference_metrics['accuracy'])
    if previous is not None:
        summary['previous_reported_accuracy']=previous['accuracy']
    atomic_json(f'artifacts/{prefix}_final_results.json',summary)
    write_report(plan,summary)
    update_status(STATUS,dict(status='complete',selected_model=selected['name'],
                  validation_accuracy=selected_metrics['accuracy'],validation_macro_f1=selected_metrics['macro_f1'],
                  predictions=summary['predictions']),required=True)
    print(json.dumps(dict(event='complete',selected=selected['name'],accuracy=selected_metrics['accuracy'],
                         macro_f1=selected_metrics['macro_f1'],gain_pp=summary['accuracy_gain_percentage_points'])),flush=True)


def write_report(plan,summary):
    metric=summary['selected_validation_metrics']
    lines=['# '+plan.get('report_title','监督微调与准确率优化结果'),'',
           f'最佳推理方案：**{summary["selected_model"]}**。验证 Accuracy **{metric["accuracy"]:.2%}**，Macro-F1 **{metric["macro_f1"]:.2%}**。',
           f'相对同精度复核的原 ConvNeXt 基线，准确率变化 **{summary["accuracy_gain_percentage_points"]:+.2f} 个百分点**。','',
           '| 单模型 | 类别 | 最佳轮次 | 已训轮数 | 验证 Accuracy | Macro-F1 |',
           '| --- | --- | ---: | ---: | ---: | ---: |']
    for row in sorted(summary['runs'],key=lambda r:(r['accuracy'],r['macro_f1']),reverse=True):
        lines.append(f'| {row["name"]} | {row["phase"]} | {row["best_epoch"]} | {row["epochs_completed"]} | {row["accuracy"]:.2%} | {row["macro_f1"]:.2%} |')
    lines+=['',f'相对基准融合（{summary["reference_ensemble_metrics"]["accuracy"]:.2%}），准确率变化 **{summary["gain_over_reference_ensemble_pp"]:+.2f} 个百分点**。',
            '上表统一重载权重，以 FP32、无翻转进行验证。最佳轮次 0 表示保留微调前的初始权重。','',
            f'最佳单模型推理方案：`{summary["best_single"]["name"]}`，Accuracy {summary["best_single"]["accuracy"]:.2%}。','',
            '最终方案组成：','']
    for item in summary['selected_recipe']['components']:
        lines.append(f'- `{item["run"]}`：权重 {item["weight"]:.3f}，水平翻转 TTA {"启用" if item["flip_tta"] else "关闭"}。')
    lines+=['','| 类别 | Precision | Recall | F1 | 样本数 |','| --- | ---: | ---: | ---: | ---: |']
    for name in summary['selected_recipe']['classes']:
        row=metric['per_class'][name]
        lines.append(f'| {name} | {row["precision"]:.2%} | {row["recall"]:.2%} | {row["f1-score"]:.2%} | {int(row["support"])} |')
    if plan.get('result_prefix')=='transfer':
        by_name={row['name']:row for row in summary['runs']}
        ce=by_name['transfer_emotieff_b0_ce_s42']
        eac=by_name['transfer_emotieff_b0_eac_s42']
        lines+=['',f'B0 配对对照：增加 EAC 后单模型 Accuracy 变化 {100*(eac["accuracy"]-ce["accuracy"]):+.2f} 个百分点。仅一个种子，不能据此判断统计显著性。',
                '', '| 重点类别 | 原融合 Recall | 本轮最终 Recall | Recall 变化（百分点） |',
                '| --- | ---: | ---: | ---: |']
        for name in ('Fear','Sad'):
            before=summary['reference_ensemble_metrics']['per_class'][name]['recall']
            after=metric['per_class'][name]['recall']
            lines.append(f'| {name} | {before:.2%} | {after:.2%} | {100*(after-before):+.2f} |')
    if plan.get('replacement_parents'):
        lines+=['','本阶段各新模型的最佳轮次按替换对应组件后的四模型融合 Accuracy 选择；上表记录该轮次的单模型成绩，并不一定是单模型最高分。',
                'B2 CE 与 B2 EAC 使用相同起点、种子和训练预算；单个种子的差异不代表统计显著性。']
    lines+=['','全部 7,065 张无标签测试图已生成预测，并检查文件名唯一性、类别映射和概率归一化。',
            f'预测：[best_predictions.csv](../{summary["predictions"]})；推理配置：[inference_recipe.json](../{plan.get("prediction_directory","predictions/sft")}/inference_recipe.json)。',
            '本轮验证集同时用于调参、早停和选择融合方案，因此成绩是开发验证结果；独立测试准确率仍需真实测试标签。',
            '完整配置与方法参考：['+plan.get('plan_document','SFT_PLAN.md')+']('+plan.get('plan_document','SFT_PLAN.md')+')；'
            f'详细候选分数：[{plan.get("result_prefix","sft")}_final_results.json](../artifacts/{plan.get("result_prefix","sft")}_final_results.json)。',
            f'生成时间（北京时间）：{datetime.now(timezone(timedelta(hours=8))).isoformat()}']
    with atomic_output(plan['report']) as temp:
        temp.write_text('\\n'.join(lines)+'\\n',encoding='utf-8')


if __name__=='__main__':
    try:
        main()
    except BaseException as error:
        update_status(STATUS,dict(status='failed',error=repr(error)))
        traceback.print_exc()
        raise
""",

'fer/finalize.py': """\
\"\"\"Audit completed validation runs, select a model, and predict the unlabeled test set.\"\"\"
import argparse
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import time
import traceback

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import FERDataset, ImagePipeline
from .io import atomic_json, read_json, update_status
from .models import make_model
from .train import classification_metrics, evaluate, seed_everything


def completed_configs(plan):
    configs=[read_json(path) for path in plan['configs']]
    return all((Path(c['output'])/'status.json').exists()
               and read_json(Path(c['output'])/'status.json').get('status')=='complete' for c in configs)


def write_progress(plan):
    lines=['# FER 实验进行中','',
           '下表是训练中的最佳验证成绩，完整对照尚未结束。每约 20 秒更新，重新打开文件可查看。','',
           '| 配置 | 状态 | 轮次 | 当前最佳 Accuracy | 当前最佳 Macro-F1 |',
           '| --- | --- | --- | ---: | ---: |']
    for path in plan['configs']:
        config=read_json(path)
        run=Path(config['output'])
        status=read_json(run/'status.json') if (run/'status.json').exists() else {}
        metrics=read_json(run/'best_metrics.json') if (run/'best_metrics.json').exists() else None
        accuracy=f'{metrics["accuracy"]:.2%}' if metrics else '—'
        f1=f'{metrics["macro_f1"]:.2%}' if metrics else '—'
        epoch=status.get('epoch',config['epochs'] if status.get('status')=='complete' else 0)
        lines.append(f'| {config["name"]} | {status.get("status","pending")} | {epoch}/{config["epochs"]} | {accuracy} | {f1} |')
    lines.extend(['','统一验证集 5,409 张，原 test 无标签；没有正式测试准确率。',
                  '全部训练完成后，将自动重载最佳权重、汇总每类指标并生成测试图预测。','',
                  '配置、数据排除规则与复现说明见 [README.md](README.md)。',
                  f'更新时间（UTC）：{datetime.now(timezone.utc).isoformat()}'])
    Path('experiments/RESULTS.md').write_text('\\n'.join(lines)+'\\n',encoding='utf-8')


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--plan',default='experiments/plan.json')
    parser.add_argument('--wait-for-queue',action='store_true')
    args=parser.parse_args()
    plan=read_json(args.plan)
    status_path=Path('artifacts/finalization_status.json')
    if args.wait_for_queue:
        print('Waiting for all four training runs to complete.',flush=True)
        while not completed_configs(plan):
            queue=read_json('artifacts/queue_status.json') if Path('artifacts/queue_status.json').exists() else {}
            state='waiting_for_failed_run_to_resume' if queue.get('status')=='failed' else 'waiting_for_training'
            update_status(status_path,dict(status=state))
            write_progress(plan)
            time.sleep(20)
    elif not completed_configs(plan):
        raise RuntimeError('All planned training runs must finish before final evaluation.')
    update_status(status_path,dict(status='auditing_validation'))
    results=[]
    for path in plan['configs']:
        config=read_json(path)
        run=Path(config['output'])
        metrics=read_json(run/'best_metrics.json')
        split=read_json(Path(config['data'])/'split.json')
        saved=np.load(run/'val_predictions.npz')
        np.testing.assert_array_equal(saved['indices'],split['val'])
        recalculated=classification_metrics(saved['labels'],saved['probabilities'],split['classes'])
        assert abs(recalculated['accuracy']-metrics['accuracy'])<1e-12
        assert abs(recalculated['macro_f1']-metrics['macro_f1'])<1e-12
        history=[json.loads(line) for line in (run/'metrics.jsonl').read_text(encoding='utf-8').splitlines()]
        assert len(history)==config['epochs']
        initialization=read_json(run/'initialization.json')
        results.append(dict(name=config['name'],model=config['model'],config_path=path,
                            epoch=metrics['epoch'],accuracy=metrics['accuracy'],macro_f1=metrics['macro_f1'],
                            per_class=metrics['per_class'],confusion_matrix=metrics['confusion_matrix'],
                            parameters=initialization['parameters'],pretraining=initialization['pretrained']['pretraining'],
                            training_seconds=sum(row['epoch_seconds'] for row in history),
                            peak_memory_gib=max(row['peak_gpu_memory_gib'] for row in history)))
    winner=max(results,key=lambda row:(row['macro_f1'],row['accuracy']))
    config=read_json(winner['config_path'])
    run=Path(config['output'])
    seed_everything(config['seed'])
    model,_=make_model(config,load_pretrained=False)
    checkpoint=torch.load(run/'best.pt',map_location='cpu',weights_only=True)
    model.load_state_dict(checkpoint['model'],strict=True)
    model=model.to(device='cuda',memory_format=torch.channels_last).eval()
    pipeline=ImagePipeline('cuda')
    validation=FERDataset(config['data'],'val')
    val_loader=DataLoader(validation,batch_size=config['eval_batch_size'],num_workers=0,pin_memory=True)
    update_status(status_path,dict(status='checking_winner',winner=winner['name']))
    y,probs,indices=evaluate(model,val_loader,pipeline,config['model'],'cuda',tta=False)
    reloaded=classification_metrics(y,probs,validation.split['classes'])
    saved=np.load(run/'val_predictions.npz')
    max_delta=float(np.max(np.abs(probs-saved['probabilities'])))
    if abs(reloaded['accuracy']-winner['accuracy'])>.001 or abs(reloaded['macro_f1']-winner['macro_f1'])>.001:
        raise RuntimeError('Reloaded checkpoint differs materially from saved validation predictions.')
    _,tta_probs,_=evaluate(model,val_loader,pipeline,config['model'],'cuda',tta=True)
    tta_metrics=classification_metrics(y,tta_probs,validation.split['classes'])
    use_tta=(tta_metrics['macro_f1'],tta_metrics['accuracy'])>(reloaded['macro_f1'],reloaded['accuracy'])
    selected=tta_metrics if use_tta else reloaded
    output=Path('predictions')
    output.mkdir(exist_ok=True)
    np.savez_compressed(output/'selected_validation_predictions.npz',labels=y,probabilities=tta_probs if use_tta else probs,indices=indices)
    testing=FERDataset(config['data'],'test')
    test_loader=DataLoader(testing,batch_size=config['eval_batch_size'],num_workers=0,pin_memory=True)
    update_status(status_path,dict(status='predicting_test',winner=winner['name'],flip_tta=use_tta))
    _,test_probs,test_indices=evaluate(model,test_loader,pipeline,config['model'],'cuda',tta=use_tta)
    assert test_probs.shape==(7065,6)
    assert np.all(np.isfinite(test_probs))
    np.testing.assert_allclose(test_probs.sum(1),1.,atol=1e-5)
    classes=testing.split['classes']
    with (output/'best_model_predictions.csv').open('w',encoding='utf-8',newline='') as file:
        writer=csv.writer(file)
        writer.writerow(['image','predicted_label','class_id','confidence'])
        for index,probability in zip(test_indices,test_probs):
            label=int(probability.argmax())
            writer.writerow([Path(testing.rows[int(index)]['path']).name,classes[label],label,f'{float(probability[label]):.8f}'])
    np.savez_compressed(output/'test_probabilities.npz',probabilities=test_probs,indices=test_indices,classes=np.array(classes))
    atomic_json(output/'label_mapping.json',{str(i):name for i,name in enumerate(classes)})
    summary=dict(completed_at=datetime.now(timezone.utc).isoformat(),selection_metric='validation_macro_f1',
                 validation_images=len(validation),test_images=len(testing),test_labels_available=False,
                 manifest_sha256=validation.split['manifest_sha256'],runs=results,
                 selected_model=winner['name'],selected_epoch=winner['epoch'],flip_tta=use_tta,
                 selected_validation_metrics=selected,tta_validation_metrics=tta_metrics,
                 checkpoint_reload_max_probability_delta=max_delta,
                 best_checkpoint=str(run/'best.pt'),predictions=str(output/'best_model_predictions.csv'),
                 limitations=['One split and one training seed; 12-epoch initial comparison, not a convergence claim.',
                              'NLA is transferred to EmoNeXt; QCS uses different pretraining and compute.',
                              'Validation is used for checkpoint/model/TTA selection; no independent labeled test result.'])
    atomic_json('artifacts/final_results.json',summary)
    lines=['# 第一轮本机 FER 实验结果','',
           '所有模型完成 12 轮，使用同一固定验证集。表中为验证集选出的最佳轮次，未使用无标签测试集选模。','',
           '| 模型 | Accuracy | Macro-F1 | 最佳轮次 | 参数量 | 训练+验证时间 |',
           '| --- | ---: | ---: | ---: | ---: | ---: |']
    for row in sorted(results,key=lambda row:row['macro_f1'],reverse=True):
        lines.append(f'| {row["name"]} | {row["accuracy"]:.2%} | {row["macro_f1"]:.2%} | {row["epoch"]} | {row["parameters"]/1e6:.2f}M | {row["training_seconds"]/60:.1f} 分钟 |')
    lines.extend(['',f'按 Macro-F1 选择 `{winner["name"]}`。水平翻转 TTA {"启用" if use_tta else "未启用"}；最终验证 Accuracy **{selected["accuracy"]:.2%}**，Macro-F1 **{selected["macro_f1"]:.2%}**。','',
                  '已为全部 7,065 张原测试图生成预测。其真实标签不可用，因此这里没有正式测试准确率；CSV 的标签编号是本项目内部映射，尚未核对外部提交格式。','',
                  '| 类别 | Precision | Recall | F1 | 验证样本数 |','| --- | ---: | ---: | ---: | ---: |'])
    for name in classes:
        values=selected['per_class'][name]
        lines.append(f'| {name} | {values["precision"]:.2%} | {values["recall"]:.2%} | {values["f1-score"]:.2%} | {int(values["support"])} |')
    lines.extend(['','训练曲线与混淆矩阵（若绘图环境可用）：','',
                  '![训练曲线](../artifacts/figures/learning_curves.png)','',
                  '![混淆矩阵](../artifacts/figures/confusion_matrix.png)','',
                  '这是一轮单划分、单种子的初步实验。NLA 是在 EmoNeXt 上的损失迁移，QCS 的预训练和计算预算也不同；不能据此宣称方法普遍优劣或达到公开 SOTA。','',
                  f'权重：`{run.as_posix()}/best.pt`；预测：`predictions/best_model_predictions.csv`；详细指标：`artifacts/final_results.json`。','',
                  '模型重载后再次计算验证指标，并检查保存的逐样本预测、标签映射、完整测试文件数和概率归一化。'])
    Path('experiments/RESULTS.md').write_text('\\n'.join(lines)+'\\n',encoding='utf-8')
    update_status(status_path,dict(status='complete',selected_model=winner['name'],
                                validation_accuracy=selected['accuracy'],validation_macro_f1=selected['macro_f1'],
                                completed_at=datetime.now(timezone.utc).isoformat()),required=True)
    print(json.dumps(summary,ensure_ascii=True),flush=True)


if __name__=='__main__':
    try:
        main()
    except Exception as error:
        update_status('artifacts/finalization_status.json',dict(status='failed',error=repr(error),
                    updated_at=datetime.now(timezone.utc).isoformat()))
        traceback.print_exc()
        raise SystemExit(1)
""",
})


In [ ]:
write_sources({
'scripts/download_weights.py': """\
\"\"\"Download only the public pretrained weights cited by the selected projects.\"\"\"
import argparse
import hashlib
import json
from pathlib import Path
import time
import urllib.request

SOURCES = {
    'emotieff_b0': ('https://raw.githubusercontent.com/sb-ai-lab/EmotiEffLib/520a051c64cd191521e5934655314e769a319684/models/affectnet_emotions/enet_b0_8_best_vgaf.pt',
                    'enet_b0_8_best_vgaf.pt', 'https://github.com/sb-ai-lab/EmotiEffLib/tree/520a051c64cd191521e5934655314e769a319684'),
    'emotieff_b2': ('https://raw.githubusercontent.com/sb-ai-lab/EmotiEffLib/520a051c64cd191521e5934655314e769a319684/models/affectnet_emotions/enet_b2_7.pt',
                    'enet_b2_7.pt', 'https://github.com/sb-ai-lab/EmotiEffLib/tree/520a051c64cd191521e5934655314e769a319684'),
    'convnext_small22k': ('https://dl.fbaipublicfiles.com/convnext/convnext_small_22k_224.pth',
                         'convnext_small_22k_224.pth', 'https://github.com/facebookresearch/ConvNeXt'),
    'ir50': ('https://drive.usercontent.google.com/download?id=1FV8kUSeVbZ815iWt-YIYiQrCDChrhO2G&export=download&authuser=0&confirm=t',
             'ir50_msceleb.pth', 'https://github.com/birdwcp/QCS'),
    'convnext22k': ('https://dl.fbaipublicfiles.com/convnext/convnext_tiny_22k_224.pth',
                    'convnext_tiny_22k_224.pth', 'https://github.com/yelboudouri/EmoNeXt'),
}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--name', choices=list(SOURCES), required=True)
    args = parser.parse_args()
    url, filename, provenance = SOURCES[args.name]
    directory = Path(__file__).resolve().parents[1] / 'artifacts' / 'weights'
    directory.mkdir(parents=True, exist_ok=True)
    dest = directory / filename
    if dest.exists():
        print(f'Already exists: {dest.name} ({dest.stat().st_size} bytes)', flush=True)
        return
    temp = dest.with_suffix('.part')
    request = urllib.request.Request(url, headers={'User-Agent': 'FER-research-weight-download/1.0'})
    digest = hashlib.sha256()
    total = 0
    started = time.time()
    with urllib.request.urlopen(request, timeout=45) as response:
        content_type = response.headers.get('Content-Type', '')
        if 'text/html' in content_type:
            raise RuntimeError('Source returned an HTML page instead of a checkpoint.')
        with temp.open('wb') as file:
            while chunk := response.read(4*1024*1024):
                file.write(chunk)
                digest.update(chunk)
                total += len(chunk)
                print(f'{filename}: {total/2**20:.1f} MiB', flush=True)
    if total < 10*1024*1024:
        raise RuntimeError(f'Checkpoint unexpectedly small: {total}; partial file retained for diagnosis.')
    temp.replace(dest)
    metadata = dict(source=url, provenance=provenance, filename=filename, bytes=total,
                    sha256=digest.hexdigest(), seconds=round(time.time()-started,2))
    dest.with_suffix('.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(json.dumps(metadata, indent=2), flush=True)


if __name__ == '__main__':
    main()
""",

'scripts/finish_experiments.py': """\
\"\"\"Wait for training, finalize predictions and render plots using available runtimes.\"\"\"
import json
import os
from pathlib import Path
import subprocess
import sys
import traceback

ROOT=Path(__file__).resolve().parents[1]


def main():
    os.chdir(ROOT)
    env=os.environ.copy()
    env['PYTHONDONTWRITEBYTECODE']='1'
    env['PYTHONUTF8']='1'
    flags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0
    with (ROOT/'artifacts'/'finalization_console.log').open('a',encoding='utf-8') as log:
        result=subprocess.run([sys.executable,'-B','-u','-m','fer.finalize','--wait-for-queue'],
                              stdout=log,stderr=subprocess.STDOUT,env=env,cwd=ROOT,creationflags=flags)
        if result.returncode:
            print(f'Final evaluation failed; see artifacts/finalization_console.log ({result.returncode})',flush=True)
            raise SystemExit(result.returncode)
        plot_python=Path.home()/'AppData/Local/Programs/Python/Python313/python.exe'
        plot=subprocess.run([str(plot_python),'-B','-u','scripts/plot_results.py'],
                            stdout=log,stderr=subprocess.STDOUT,env=env,cwd=ROOT,creationflags=flags)
        print(f'Experiment results ready. Plot exit code: {plot.returncode}',flush=True)


if __name__=='__main__':
    main()
""",

'scripts/plot_results.py': """\
\"\"\"Render static scientific plots from completed, audited experiment results.\"\"\"
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np


def main():
    root=Path(__file__).resolve().parents[1]
    result=json.loads((root/'artifacts/final_results.json').read_text(encoding='utf-8'))
    output=root/'artifacts/figures'
    output.mkdir(parents=True,exist_ok=True)
    fig,axes=plt.subplots(1,2,figsize=(12,4.3),layout='constrained')
    for run in result['runs']:
        rows=[json.loads(line) for line in (root/'runs'/run['name']/'metrics.jsonl').read_text(encoding='utf-8').splitlines()]
        epochs=[row['epoch'] for row in rows]
        axes[0].plot(epochs,[100*row['val_accuracy'] for row in rows],marker='.',label=run['name'])
        axes[1].plot(epochs,[100*row['val_macro_f1'] for row in rows],marker='.',label=run['name'])
    for axis,title in zip(axes,['Validation accuracy (%)','Validation macro-F1 (%)']):
        axis.set(xlabel='Epoch',ylabel=title)
        axis.grid(alpha=.2)
        axis.legend(fontsize=8)
    fig.suptitle('FER: fixed duplicate-free development split, seed 42')
    fig.savefig(output/'learning_curves.png',dpi=170)
    plt.close(fig)
    labels=['Angry','Fear','Happy','Neutral','Sad','Surprise']
    counts=np.array(result['selected_validation_metrics']['confusion_matrix'])
    proportions=counts/counts.sum(1,keepdims=True)
    fig,axis=plt.subplots(figsize=(7,6),layout='constrained')
    plot=axis.imshow(proportions,cmap='Blues',vmin=0,vmax=1)
    axis.set_xticks(range(6),labels,rotation=35,ha='right')
    axis.set_yticks(range(6),labels)
    axis.set(xlabel='Predicted label',ylabel='Dataset label',title=result['selected_model']+' — validation')
    for row in range(6):
        for col in range(6):
            axis.text(col,row,f'{counts[row,col]}\\n{proportions[row,col]:.1%}',ha='center',va='center',
                      color='white' if proportions[row,col]>.55 else '#17202a',fontsize=9)
    fig.colorbar(plot,ax=axis,label='Fraction of each true class',shrink=.8)
    fig.savefig(output/'confusion_matrix.png',dpi=170)
    plt.close(fig)
    print('Saved learning_curves.png and confusion_matrix.png',flush=True)


if __name__=='__main__':
    main()
""",

'scripts/prepare_emotieff.py': """\
\"\"\"Convert two pinned official models into audited tensor-only checkpoints.\"\"\"
import hashlib
import importlib
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
import timm
import torch
from fer.emotieff import CLASS_ROWS, SOURCE_CLASSES, TARGET_CLASSES, VARIANTS, map_classifier
from fer.io import atomic_json
from fer.train import atomic_checkpoint


ALLOWED = {
    'timm.layers.adaptive_avgmax_pool.SelectAdaptivePool2d',
    'timm.layers.conv2d_same.Conv2dSame', 'timm.layers.norm_act.BatchNormAct2d',
    'timm.models._efficientnet_blocks.DepthwiseSeparableConv',
    'timm.models._efficientnet_blocks.InvertedResidual',
    'timm.models._efficientnet_blocks.SqueezeExcite',
    'timm.models.efficientnet.EfficientNet',
    'torch.nn.modules.activation.SiLU', 'torch.nn.modules.activation.Sigmoid',
    'torch.nn.modules.container.Sequential', 'torch.nn.modules.conv.Conv2d',
    'torch.nn.modules.flatten.Flatten', 'torch.nn.modules.linear.Identity',
    'torch.nn.modules.linear.Linear', 'torch.nn.modules.pooling.AdaptiveAvgPool2d',
    'torch.nn.modules.pooling.AdaptiveMaxPool2d',
}
SOURCES = {
    'b0': ('enet_b0_8_best_vgaf.pt', '95aafb39b8bb87964f45e208b9ab31e276e3e5278678db4961d18e6a1b42a141'),
    'b2': ('enet_b2_7.pt', 'e687fd9e7bee45486c7325f11222740f841c3bebac531badb903a50e82b9ac0f'),
}


def main():
    torch.set_num_threads(4)
    torch.manual_seed(42)
    directory = ROOT / 'artifacts/weights'
    for variant, (filename, expected) in SOURCES.items():
        source = directory / filename
        if hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Official checkpoint hash changed: {filename}')
        names = torch.serialization.get_unsafe_globals_in_checkpoint(source)
        if not set(names) <= ALLOWED:
            raise RuntimeError(f'Unreviewed serialized types: {set(names) - ALLOWED}')
        objects = [getattr(importlib.import_module(n.rsplit('.', 1)[0]), n.rsplit('.', 1)[1]) for n in names]
        with torch.serialization.safe_globals(objects):
            original = torch.load(source, map_location='cpu', weights_only=True).eval()
        # timm 1.0 added optional space-to-depth and antialias modules. These
        # old checkpoints have neither operation; restore only their no-op defaults.
        compatibility = []
        for name, module in original.named_modules():
            if type(module).__name__ in ('DepthwiseSeparableConv','InvertedResidual'):
                for attribute in ('conv_s2d','bn_s2d'):
                    if not hasattr(module,attribute):
                        setattr(module,attribute,None)
                        compatibility.append(name+'.'+attribute+'=None')
                if not hasattr(module,'aa'):
                    module.aa = torch.nn.Identity()
                    compatibility.append(name+'.aa=Identity')
        spec = VARIANTS[variant]
        state = map_classifier(original.state_dict(), spec['source_classes'])
        recreated = timm.create_model(spec['architecture'], pretrained=False, num_classes=6,
                                      global_pool=spec['pool'], drop_rate=0., drop_path_rate=0.).eval()
        recreated.load_state_dict(state, strict=True)
        with torch.inference_mode():
            inputs = torch.randn(2, 3, spec['image_size'], spec['image_size'])
            expected_logits = original(inputs)[:, CLASS_ROWS[spec['source_classes']]]
            actual = recreated(inputs)
            torch.testing.assert_close(actual, expected_logits, rtol=2e-5, atol=2e-5)
        metadata = dict(variant=variant, **spec, source_file=filename, source_sha256=expected,
                        source_url=f'https://raw.githubusercontent.com/sb-ai-lab/EmotiEffLib/520a051c64cd191521e5934655314e769a319684/models/affectnet_emotions/{filename}',
                        classes=TARGET_CLASSES, original_classes=SOURCE_CLASSES[spec['source_classes']],
                        classifier_rows=CLASS_ROWS[spec['source_classes']], timm=timm.__version__,
                        torch=str(torch.__version__), max_logit_error=float((actual-expected_logits).abs().max()),
                        compatibility_noops=compatibility,
                        verification='strict state load and full-resolution FP32 logit equivalence')
        output = directory / f'emotieff_{variant}_six_classes.pt'
        atomic_checkpoint(output, dict(model=state, metadata=metadata))
        metadata['converted_sha256'] = hashlib.sha256(output.read_bytes()).hexdigest()
        atomic_json(output.with_suffix('.json'), metadata)
        print(metadata, flush=True)


if __name__ == '__main__':
    main()
""",

'scripts/prepare_refine.py': """\
\"\"\"Pin the current best recipe and prepare a bounded component refinement plan.\"\"\"
import copy
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np
from fer.data import FERDataset
from fer.io import atomic_json, read_json
from fer.refinement import ReplacementReference, file_sha256


def main():
    plan_path = ROOT/'experiments/refine_plan.json'
    if plan_path.exists():
        raise FileExistsError('Refinement plan already exists; inspect rather than overwrite.')
    previous = read_json('artifacts/transfer_final_results.json')
    recipe = previous['selected_recipe']
    validation = FERDataset('artifacts/data','val')
    assert recipe['classes'] == validation.split['classes']
    assert recipe['manifest_sha256'] == file_sha256('artifacts/data/manifest.jsonl')
    assert len(recipe['components']) == 4
    references, arrays = [], {}
    for component in recipe['components']:
        assert component['weight'] == .25 and not component['flip_tta']
        name = component['run']
        assert component['checkpoint_sha256'] == file_sha256(component['checkpoint'])
        npz = f'artifacts/transfer_validation/{name}.npz'
        meta = f'artifacts/transfer_validation/{name}.json'
        row = dict(name=name,weight=.25,checkpoint=component['checkpoint'],
                   checkpoint_sha256=component['checkpoint_sha256'],probabilities=npz,
                   probabilities_sha256=file_sha256(npz),metadata=meta,metadata_sha256=file_sha256(meta))
        references.append(row)
        with np.load(npz,allow_pickle=False) as saved:
            np.testing.assert_array_equal(saved['indices'],validation.indices)
            np.testing.assert_array_equal(saved['labels'],validation.labels)
            arrays[name] = saved['probabilities'].copy()
    baseline = sum(arrays[row['name']]*row['weight'] for row in references)
    accuracy = float(np.mean(baseline.argmax(1)==validation.labels))
    assert accuracy == previous['selected_validation_metrics']['accuracy']

    def selection_spec(parent):
        replaced = next(row for row in references if row['name']==parent)
        spec = dict(candidate_weight=.25,components=[r for r in references if r['name']!=parent],
                    replaces_checkpoint=replaced['checkpoint'],
                    replaces_checkpoint_sha256=replaced['checkpoint_sha256'],
                    manifest_sha256=recipe['manifest_sha256'],classes=recipe['classes'],baseline_accuracy=accuracy)
        scorer=ReplacementReference(spec,validation.indices,validation.labels,recipe['classes'],recipe['manifest_sha256'])
        recombined=scorer.combine(validation.labels,arrays[parent],validation.indices)
        np.testing.assert_array_equal(recombined.argmax(1),baseline.argmax(1))
        return spec

    b2parent='transfer_emotieff_b2_ce_s42'
    emoparent='emonext_tiny_s42'
    b2=read_json(f'experiments/configs/{b2parent}.json')
    b2.update(name='refine_b2_ce_s52',output='runs/refine_b2_ce_s52',seed=52,epochs=14,
              lr=8e-6,head_lr_multiplier=2.,weight_decay=5e-4,warmup_epochs=1,
              early_stopping_patience=4,min_epochs=5,finetune_from=f'runs/{b2parent}/best.pt',
              selection_reference=selection_spec(b2parent))
    eac=copy.deepcopy(b2)
    eac.update(name='refine_b2_eac_s52',output='runs/refine_b2_eac_s52',loss='eac',eac_weight=.05)
    emo=read_json(f'experiments/configs/{emoparent}.json')
    emo.update(name='refine_emonext_s53',output='runs/refine_emonext_s53',seed=53,epochs=12,
               lr=1.5e-5,weight_decay=.02,label_smoothing=.15,warmup_epochs=1,
               early_stopping_patience=4,min_epochs=5,image_size=224,
               selection_metric='validation_accuracy',eval_precision='float32',eval_batch_size=32,
               evaluate_initial=True,finetune_from=f'runs/{emoparent}/best.pt',
               freeze_stages=1,freeze_modules=['localization','fc_loc'],drop_path_rate=.2,layer_decay=.75,
               mixup_alpha=.2,mixup_probability=.5,
               augmentation=dict(degrees=10,scale_range=.05,translation=.03,jitter=.08,erase_probability=.25),
               selection_reference=selection_spec(emoparent))
    paths=[]
    for config in (b2,eac,emo):
        path=f'experiments/configs/{config["name"]}.json'
        atomic_json(path,config)
        paths.append(path)
    plan=dict(phase='best_ensemble_component_refinement',configs=paths,
              reference_configs=[c['config_path'] for c in recipe['components']],comparison_configs=[],
              data='artifacts/data',seed=20261014,eval_precision='float32',ensemble_weights=[],
              reference_blend_weights=[],replacement_parents={b2['name']:b2parent,eac['name']:b2parent,emo['name']:emoparent},
              result_prefix='refine',prediction_directory='predictions/refine',
              report='experiments/REFINE_RESULTS.md',report_title='最佳融合模型继续微调',
              plan_document='REFINE_PLAN.md',queue_status='artifacts/refine_queue_status.json',
              controller_lock='artifacts/refine_controller.lock',finalization_log='artifacts/refine_finalization_console.log',
              previous_result='artifacts/transfer_final_results.json',
              progress_notes=['基准为四模型等权融合，验证 Accuracy 72.3794%。',
                              '新模型按替换相应组件后的四模型融合 Accuracy 保存最佳权重；Macro-F1 用于同分比较。',
                              'B2 CE / EAC 配对低学习率微调；EmoNeXt 增加 Mixup、随机深度并冻结第一阶段与 STN。',
                              '最终保留原四模型候选，比较三个单组件替换和两个双组件替换；全部新输出放在 refine 目录。'])
    atomic_json(plan_path,plan)
    atomic_json('artifacts/refine_reference_audit.json',dict(baseline_accuracy=accuracy,classes=recipe['classes'],
                manifest_sha256=recipe['manifest_sha256'],validation_images=len(validation),references=references,
                previous_result_sha256=file_sha256('artifacts/transfer_final_results.json'),
                existing_submission_sha256=file_sha256('submission_best_7238.csv')))
    print(f'Prepared {len(paths)} refinement trials; reference accuracy {accuracy:.8%}.',flush=True)


if __name__=='__main__':
    main()
""",

'scripts/run_experiments.py': """\
\"\"\"Run the GPU experiments sequentially, logging and resuming at epoch boundaries.\"\"\"
import argparse
import os
from pathlib import Path
import subprocess
import sys
import time

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT))
from fer.io import read_json, update_status


def write_status(value):
    path=ROOT/'artifacts'/'queue_status.json'
    update_status(path,value,required=value['status']=='training_complete')


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--plan',default='experiments/plan.json')
    args=parser.parse_args()
    os.chdir(ROOT)
    (ROOT/'artifacts'/'controller.pid').write_text(str(os.getpid()),encoding='utf-8')
    plan=read_json(args.plan)
    env=os.environ.copy()
    env['PYTHONDONTWRITEBYTECODE']='1'
    env['PYTHONUTF8']='1'
    completed=[]
    for config_path in plan['configs']:
        config=read_json(config_path)
        output=ROOT/config['output']
        output.mkdir(parents=True,exist_ok=True)
        status_path=output/'status.json'
        if status_path.exists() and read_json(status_path).get('status')=='complete':
            completed.append(config['name'])
            continue
        command=[sys.executable,'-B','-u','-m','fer.train','--config',config_path]
        if (output/'last.pt').exists(): command.append('--resume')
        with (output/'console.log').open('a',encoding='utf-8') as log:
            child=subprocess.Popen(command,cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,
                                   creationflags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0)
            print(f'Started {config["name"]}: PID {child.pid}',flush=True)
            while child.poll() is None:
                write_status(dict(status='running',current=config['name'],pid=child.pid,
                                  completed=completed,plan=args.plan,controller_pid=os.getpid()))
                time.sleep(5)
            if child.returncode:
                write_status(dict(status='failed',current=config['name'],exit_code=child.returncode,
                                  completed=completed,log=str(output/'console.log')))
                raise SystemExit(child.returncode)
        completed.append(config['name'])
        write_status(dict(status='between_runs',completed=completed))
    write_status(dict(status='training_complete',completed=completed))
    print('All training runs completed.',flush=True)


if __name__=='__main__':
    main()
""",

'scripts/run_sft.py': """\
\"\"\"Run the bounded supervised fine-tuning plan and then select/export results.\"\"\"
import argparse
from datetime import datetime, timedelta, timezone
import os
from pathlib import Path
import subprocess
import sys
import time
import traceback

ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT))
from fer.io import atomic_output, read_json, update_status


def write_progress(plan, state):
    lines=['# '+plan.get('report_title','监督微调实验进度'),'',f'当前阶段：{state}。按固定验证集 Accuracy 选模，Macro-F1 用于同分比较。','',
           '| 模型 | 状态 | 完成轮数 / 上限 | 所选轮次单模型 Accuracy | Macro-F1 | 替换后融合 Accuracy |',
           '| --- | --- | ---: | ---: | ---: | ---: |']
    for path in plan['reference_configs']+plan.get('comparison_configs',[])+plan['configs']:
        config=read_json(path)
        run=ROOT/config['output']
        status=read_json(run/'status.json') if (run/'status.json').exists() else {}
        metrics=read_json(run/'best_metrics.json') if (run/'best_metrics.json').exists() else {}
        history=(run/'metrics.jsonl').read_text(encoding='utf-8').splitlines() if (run/'metrics.jsonl').exists() else []
        accuracy=f'{metrics["accuracy"]:.2%}' if metrics else '—'
        macro=f'{metrics["macro_f1"]:.2%}' if metrics else '—'
        fusion=f'{metrics["selection_metrics"]["accuracy"]:.2%}' if metrics and config.get('selection_reference') else '—'
        label='已完成参照' if path not in plan['configs'] else status.get('status','pending')
        lines.append(f'| {config["name"]} | {label} | {len(history)} / {config["epochs"]} | {accuracy} | {macro} | {fusion} |')
    lines+=['']+plan.get('progress_notes',['前两组从已有最佳 ConvNeXt 权重微调；第三组从 ImageNet-22K 的 ConvNeXt-Small 权重开始监督训练。'])+[
            '采用早停，完成轮数可能少于上限。最佳轮次为 0 表示初始权重仍优于后续训练。',
            '本阶段训练结束后统一以 FP32 复核各候选，再比较少量预定融合权重与翻转 TTA，导出最优方案。',
            '验证集有 5,409 张图，用于调参和选模；原测试集 7,065 张无标签。',
            '方案说明：['+plan.get('plan_document','SFT_PLAN.md')+']('+plan.get('plan_document','SFT_PLAN.md')+')。',
            f'更新时间（北京时间）：{datetime.now(timezone(timedelta(hours=8))).isoformat()}']
    try:
        with atomic_output(ROOT/plan['report']) as temp:
            temp.write_text('\\n'.join(lines)+'\\n',encoding='utf-8')
    except OSError as error:
        print(f'Progress report update skipped: {error}',flush=True)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--plan',default='experiments/sft_plan.json')
    args=parser.parse_args()
    os.chdir(ROOT)
    plan=read_json(args.plan)
    lock=(ROOT/plan.get('controller_lock','artifacts/sft_controller.lock')).open('a+b')
    if lock.tell()==0:
        lock.write(b'0')
        lock.flush()
    lock.seek(0)
    if os.name=='nt':
        import msvcrt
        try:
            msvcrt.locking(lock.fileno(),msvcrt.LK_NBLCK,1)
        except OSError as error:
            lock.close()
            raise RuntimeError('A fine-tuning controller is already running.') from error
    env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','PYTHONUTF8':'1'}
    flags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0
    completed=[]
    child=None
    current=None
    try:
        for path in plan['configs']:
            config=read_json(path)
            current=config['name']
            run=ROOT/config['output']
            run.mkdir(parents=True,exist_ok=True)
            if (run/'status.json').exists() and read_json(run/'status.json').get('status')=='complete':
                completed.append(current)
                continue
            command=[sys.executable,'-B','-u','-m','fer.train','--config',path]
            if (run/'last.pt').exists():
                command.append('--resume')
            with (run/'console.log').open('a',encoding='utf-8') as log:
                child=subprocess.Popen(command,cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,creationflags=flags)
                print(f'Started {current}: PID {child.pid}',flush=True)
                while child.poll() is None:
                    update_status(plan['queue_status'],dict(status='running',current=current,pid=child.pid,
                                  controller_pid=os.getpid(),completed=completed,plan=args.plan))
                    write_progress(plan,'训练中')
                    time.sleep(10)
                if child.returncode:
                    raise RuntimeError(f'{current} exited with {child.returncode}; see {run / "console.log"}')
            completed.append(current)
        current='finalization'
        update_status(plan['queue_status'],dict(status='finalizing',completed=completed,controller_pid=os.getpid()))
        write_progress(plan,'训练完成，正在复核和导出')
        finalization_log=plan.get('finalization_log','artifacts/sft_finalization_console.log')
        with (ROOT/finalization_log).open('a',encoding='utf-8') as log:
            child=subprocess.Popen([sys.executable,'-B','-u','-m','fer.finalize_sft','--plan',args.plan],
                                   cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,creationflags=flags)
            code=child.wait()
        if code:
            raise RuntimeError(f'Finalization failed with {code}; see {finalization_log}')
        update_status(plan['queue_status'],dict(status='complete',completed=completed,plan=args.plan),required=True)
        print('Fine-tuning, validation comparison and prediction export completed.',flush=True)
    except KeyboardInterrupt:
        if child is not None and child.poll() is None:
            child.terminate()
            child.wait()
        update_status(plan['queue_status'],dict(status='paused',current=current,completed=completed))
        raise
    except BaseException as error:
        if child is not None and child.poll() is None:
            child.terminate()
            child.wait()
        update_status(plan['queue_status'],dict(status='failed',current=current,completed=completed,error=repr(error)))
        traceback.print_exc()
        raise
    finally:
        lock.close()


if __name__=='__main__':
    main()
""",
})


## 4. 核验固定划分与四模型检查点


In [ ]:
from fer.data import FERDataset, ImagePipeline
from fer.models import make_model
from fer.train import classification_metrics, evaluate
from torch.utils.data import DataLoader

recipe = json.loads(checked_file('reference/inference_recipe.json').read_text(encoding='utf-8'))
CLASSES = recipe['classes']
OFFICIAL_LABELS = BUNDLE_MANIFEST['official_label_mapping']
assert CLASSES == ['Angry', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
assert abs(sum(item['weight'] for item in recipe['components']) - 1) < 1e-9
assert all(item['weight'] == .25 and not item['flip_tta'] for item in recipe['components'])
for name in ('images.npy', 'manifest.jsonl', 'split.json'):
    checked_file('data/' + name)
split = json.loads((BUNDLE / 'data/split.json').read_text(encoding='utf-8'))
assert file_sha256(BUNDLE / 'data/manifest.jsonl') == recipe['manifest_sha256'] == split['manifest_sha256']
rows = [json.loads(line) for line in (BUNDLE / 'data/manifest.jsonl').read_text(encoding='utf-8').splitlines()]
assert not ({rows[i]['pixel_sha256'] for i in split['train']} & {rows[i]['pixel_sha256'] for i in split['val']})
assert len(split['val']) == 5409 and len(split['test']) == 7065
testing = FERDataset(BUNDLE / 'data', 'test')
validation = FERDataset(BUNDLE / 'data', 'val')
for component in recipe['components']:
    path = checked_file('checkpoints/' + component['run'] + '.pt')
    assert file_sha256(path) == component['checkpoint_sha256']
display(pd.DataFrame(recipe['components'])[['run', 'model', 'image_size', 'weight', 'flip_tta']])

def load_component(component):
    path = checked_file('checkpoints/' + component['run'] + '.pt')
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    config = checkpoint['config']
    assert checkpoint['classes'] == CLASSES
    assert config['manifest_sha256'] == recipe['manifest_sha256']
    model, _ = make_model(config, load_pretrained=False)
    model.load_state_dict(checkpoint['model'], strict=True)
    del checkpoint
    return model.to(device=DEVICE, memory_format=torch.channels_last).eval(), config

def infer_dataset(dataset):
    probabilities = np.zeros((len(dataset), len(CLASSES)), dtype=np.float32)
    for component in recipe['components']:
        start = time.time()
        model, config = load_component(component)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                            pin_memory=DEVICE.type == 'cuda')
        pipeline = ImagePipeline(DEVICE, size=component['image_size'])
        _, probs, indices = evaluate(model, loader, pipeline, config['model'], DEVICE,
                                     tta=False, precision='float32')
        np.testing.assert_array_equal(indices, dataset.indices)
        assert probs.shape == probabilities.shape and np.isfinite(probs).all()
        probabilities += component['weight'] * probs
        print(component['run'], 'completed:', len(dataset), 'images;', round(time.time() - start, 1), 'seconds', flush=True)
        del model, pipeline, probs
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1., atol=1e-5)
    return probabilities


## 5. 重算测试推理与原版概率对照

四个模型依次加载，避免同时占用显存。概率取加权平均后才取 argmax。
保存原版参考概率用于核验跨平台变化；不会用训练集与测试集重复关系替换模型预测。


In [ ]:
reference_npz = np.load(checked_file('reference/test_probabilities.npz'), allow_pickle=False)
np.testing.assert_array_equal(reference_npz['indices'], testing.indices)
np.testing.assert_array_equal(reference_npz['classes'], CLASSES)
reference_probabilities = reference_npz['probabilities'].astype(np.float32)
test_probabilities = infer_dataset(testing) if RECOMPUTE_TEST else reference_probabilities.copy()
assert test_probabilities.shape == (7065, 6)
np.testing.assert_allclose(test_probabilities.sum(axis=1), 1., atol=1e-5)
changed = int(np.count_nonzero(test_probabilities.argmax(1) != reference_probabilities.argmax(1)))
print('Prediction differences versus saved best version:', changed)
print('Maximum probability difference:', float(np.abs(test_probabilities - reference_probabilities).max()))
np.savez_compressed(WORK / 'test_probabilities.npz', probabilities=test_probabilities,
                    indices=np.array(testing.indices), classes=np.array(CLASSES))


## 6. 固定验证集结果复核


In [ ]:
cached_validation = np.zeros((len(validation), 6), dtype=np.float32)
for component in recipe['components']:
    record = np.load(checked_file('validation/' + component['key'] + '.npz'), allow_pickle=False)
    np.testing.assert_array_equal(record['indices'], validation.indices)
    np.testing.assert_array_equal(record['labels'], validation.labels)
    cached_validation += component['weight'] * record['probabilities']
validation_probabilities = infer_dataset(validation) if RECOMPUTE_VALIDATION else cached_validation
validation_metrics = classification_metrics(validation.labels, validation_probabilities, CLASSES)
if not RECOMPUTE_VALIDATION:
    assert abs(validation_metrics['accuracy'] - BUNDLE_MANIFEST['validation_accuracy']) < 1e-12
    assert abs(validation_metrics['macro_f1'] - BUNDLE_MANIFEST['validation_macro_f1']) < 1e-12
print('Validation evidence:', 'recomputed checkpoints' if RECOMPUTE_VALIDATION else 'verified per-model saved probabilities')
print(f"Accuracy: {validation_metrics['accuracy']:.6%}; Macro-F1: {validation_metrics['macro_f1']:.6%}")
display(pd.DataFrame(validation_metrics['per_class']).T)
display(pd.DataFrame(validation_metrics['confusion_matrix'], index=CLASSES, columns=CLASSES))


## 7. 比赛标签映射、模板顺序与提交文件

比赛标签：`Angry=0, Fear=1, Happy=2, Sad=3, Surprise=4, Neutral=5`。
内部类别索引与比赛编号不同，必须先恢复类别名再映射。
按官方样例 `ID` 顺序输出，核验 7,065 行、唯一文件名、列名和标签范围。


In [ ]:
template = pd.read_csv(checked_file('reference/sample_submission.csv'))
assert template.columns.tolist() == ['ID', 'Emotion']
names = [Path(rows[i]['path']).name for i in testing.indices]
assert len(names) == len(set(names)) == 7065
assert set(template['ID']) == set(names) and template['ID'].is_unique
predicted_names = np.array(CLASSES)[test_probabilities.argmax(1)]
predicted_labels = pd.Series([OFFICIAL_LABELS[name] for name in predicted_names], index=names)
submission = template[['ID']].copy()
submission['Emotion'] = submission['ID'].map(predicted_labels).astype('int64')
assert not submission.isna().any().any()
assert set(submission['Emotion']).issubset(set(range(6)))
assert submission['ID'].tolist() == template['ID'].tolist()
output = WORK / 'submission.csv'
submission.to_csv(output, index=False, lineterminator='\r\n')
saved_submission = pd.read_csv(checked_file(f'reference/submission_best_{ACTIVE_VERSION}.csv'))
assert saved_submission['ID'].tolist() == submission['ID'].tolist()
row_differences = int((saved_submission['Emotion'] != submission['Emotion']).sum())
assert row_differences == changed
if not RECOMPUTE_TEST:
    assert file_sha256(output) == FILE_SPECS[f'reference/submission_best_{ACTIVE_VERSION}.csv']['sha256']
audit = dict(rows=len(submission), columns=submission.columns.tolist(), label_mapping=OFFICIAL_LABELS,
             selected_recipe=recipe['name'], output_sha256=file_sha256(output),
             test_inference_recomputed=RECOMPUTE_TEST,
             validation_inference_recomputed=RECOMPUTE_VALIDATION,
             differences_from_saved_best=row_differences, validation_metrics=validation_metrics,
             torch=torch.__version__, timm=timm.__version__, device=str(DEVICE))
(WORK / 'submission_audit.json').write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')
(WORK / 'inference_recipe.json').write_text(json.dumps(recipe, ensure_ascii=False, indent=2), encoding='utf-8')
print('Ready:', output, '| rows:', len(submission), '| differences:', row_differences)
display(submission.head())
display(submission['Emotion'].value_counts().sort_index().rename_axis('Emotion').to_frame('count'))


## 8. 训练配置与复现实验入口

以上默认执行最佳版本推理。完整训练实现已写入 `/kaggle/working/fer_project/fer/`，
原始实验配置保留在输入资源 `configs/` 中，可配合 `fer.train` 运行。
原训练从 ImageNet-22K / MS-Celeb-1M / AffectNet 检查点初始化；如需完整重训，
先用保留的下载脚本准备预训练权重，并将配置中的路径改成当前 Kaggle 路径。
微调阶段依赖前序模型和固定验证参考概率，不能仅重新训练最后一轮就宣称复现全部实验。

下表直接展示这次选中检查点保存的超参数；本 notebook 不自动启动新的训练或更改已选权重。


In [ ]:
config_rows = []
for component in recipe['components']:
    config = json.loads(checked_file('configs/' + component['run'] + '.json').read_text(encoding='utf-8'))
    config_rows.append({key: config.get(key) for key in
                        ['name', 'model', 'loss', 'seed', 'epochs', 'lr', 'weight_decay',
                         'label_smoothing', 'ema', 'eac_weight', 'freeze_stages', 'mixup_alpha']})
display(pd.DataFrame(config_rows))
source_hashes = {str(path.relative_to(PROJECT)): file_sha256(path)
                 for path in PROJECT.rglob('*') if path.is_file() and '__pycache__' not in path.parts}
(WORK / 'project_source_hashes.json').write_text(json.dumps(source_hashes, indent=2), encoding='utf-8')
shutil.make_archive(str(WORK / 'fer_project_source'), 'zip', root_dir=PROJECT)
print('Outputs: submission.csv, submission_audit.json, inference_recipe.json, test_probabilities.npz, fer_project_source.zip')
